# CCR-GNN Baseline for DMF

Notebook này train kiến trúc Graph theo bài báo **Every Corporation Owns Its Structure: Corporate Credit Ratings via Graph Neural Networks**. Thay vì xây graph giữa các công ty bằng kNN/temporal edges, mỗi dòng/công ty được ánh xạ thành một graph riêng: node là từng chiều feature trong corporate embedding, cạnh feature-feature được tạo bằng self-outer-product, sau đó GAT học tương tác local và graph pooling lấy thông tin global.

Output chính vẫn là `gat_val_predictions.csv` và `gat_test_predictions.csv` để notebook DMF/DCS join với LSTM theo `row_id`.

## Notebook Contract
- Input canonical trên Kaggle: `/kaggle/input/datasets/tailength/corporate-credit-rating/test/{train,val,test}.csv`.
- Output writable: `/kaggle/working/credit_rating_artifacts/`.
- DMF/DCS contract: prediction CSV giữ `row_id`, `ticker`, `rating_date`, true/pred labels, confidence, `prob_0..prob_n`, và `label_mapping.csv`.
- Graph contract: dùng CCR-GNN/C2G adapted cho dữ liệu này, gồm feature-node graph mỗi dòng, self-outer-product adjacency, 3 GAT layer, pooling mean/mean/max, concat local + global trước MLP.
- Class 0 là lớp thiểu số cần theo dõi riêng; notebook chọn checkpoint bằng score có `Class0_F2`, dùng mild class-0 weighting + margin loss nhẹ, và calibrate threshold class 0 bằng validation trước khi đánh giá/export test.


In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, auc, cohen_kappa_score,
    confusion_matrix, classification_report,
    precision_recall_fscore_support,
)

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def detect_kaggle_runtime() -> bool:
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '').strip():
        return True
    return Path('/kaggle/input').exists() and Path('/kaggle/working').exists()


IN_KAGGLE = detect_kaggle_runtime()


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    return start


PROJECT_ROOT = Path('/kaggle/working') if IN_KAGGLE else find_project_root(Path.cwd().resolve())
ARTIFACT_DIR = PROJECT_ROOT / 'credit_rating_artifacts'
DMF_ARTIFACT_DIR = ARTIFACT_DIR / 'dmf_gat_lstm'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DMF_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', device)
print('Project root:', PROJECT_ROOT)
print('DMF artifact dir:', DMF_ARTIFACT_DIR)


In [ ]:
FINANCIAL_FEATURES = [
    'current_ratio', 'debt_equity_ratio', 'gross_profit_margin', 'operating_profit_margin',
    'ebit_margin', 'pretax_profit_margin', 'net_profit_margin', 'asset_turnover',
    'roe', 'roa', 'operating_cashflow_ps', 'free_cashflow_ps'
]
TARGET_COL = 'rating_detail'
TARGET_ORDERED_LABELS = ['Distressed', 'HY', 'IG']


def resolve_split_path(default_path, local_fallbacks=None):
    candidates = [Path(default_path)]
    for p in (local_fallbacks or []):
        p_obj = Path(p)
        candidates.append(PROJECT_ROOT / p_obj if not p_obj.is_absolute() else p_obj)
    if IN_KAGGLE:
        kaggle_root = Path('/kaggle/input')
        expanded = []
        for p in candidates:
            expanded.append(p)
            if not p.exists() and kaggle_root.exists():
                expanded.extend(kaggle_root.rglob(p.name))
        candidates = expanded
    seen = set()
    deduped = []
    for p in candidates:
        p = Path(p)
        key = str(p)
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    for p in deduped:
        if p.exists():
            return p
    raise FileNotFoundError(f'Khong tim thay file split: {deduped}')


TRAIN_PATH = resolve_split_path(
    '/kaggle/input/datasets/tailength/corporate-credit-rating/train_augmented_timegan.csv',
    [
        'data/processed/train_augmented_timegan.csv',
        'data/external/timegan_3groups_output/splits/train_augmented_timegan.csv',
        'data/processed/test/train.csv',
    ],
)
VAL_PATH = resolve_split_path(
    '/kaggle/input/datasets/tailength/corporate-credit-rating/val.csv',
    ['data/processed/test/val.csv'],
)
TEST_PATH = resolve_split_path(
    '/kaggle/input/datasets/tailength/corporate-credit-rating/test.csv',
    ['data/processed/test/test.csv'],
)


def load_split(path, split_name):
    frame = pd.read_csv(path)
    frame = frame.copy()
    frame['__split__'] = split_name
    frame['__split_row_index__'] = np.arange(len(frame), dtype=int)
    if 'row_id' not in frame.columns:
        frame['row_id'] = [f'{split_name}_{i:06d}' for i in range(len(frame))]
    else:
        frame['row_id'] = frame['row_id'].astype(str)
    return frame


train_df = load_split(TRAIN_PATH, 'train')
val_df = load_split(VAL_PATH, 'val')
test_df = load_split(TEST_PATH, 'test')
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

split_contract = {
    'train_path': str(TRAIN_PATH),
    'val_path': str(VAL_PATH),
    'test_path': str(TEST_PATH),
    'train_rows': int(len(train_df)),
    'val_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
    'row_id_rule': 'existing row_id if present, otherwise <split>_<zero_padded_original_split_index>',
}
print('DMF split contract:', split_contract)

df = df.dropna(subset=[TARGET_COL]).copy()
target_as_num = pd.to_numeric(df[TARGET_COL], errors='coerce')
if target_as_num.notna().all():
    df[TARGET_COL] = target_as_num.astype(int)
    observed = sorted(df[TARGET_COL].unique().tolist())
    raw_to_id = {int(v): i for i, v in enumerate(observed)}
    id_to_raw = {i: int(v) for v, i in raw_to_id.items()}
    df[TARGET_COL] = df[TARGET_COL].map(raw_to_id).astype(int)
else:
    tgt = df[TARGET_COL].astype(str).str.strip()
    observed = sorted(tgt.unique().tolist())
    ordered = [x for x in TARGET_ORDERED_LABELS if x in observed] if set(observed).issubset(set(TARGET_ORDERED_LABELS)) else observed
    raw_to_id = {v: i for i, v in enumerate(ordered)}
    id_to_raw = {i: v for i, v in raw_to_id.items()}
    df[TARGET_COL] = tgt.map(raw_to_id).astype(int)

n_classes = int(df[TARGET_COL].nunique())
label_contract = pd.DataFrame({
    'label_id': list(range(n_classes)),
    'label_name': [str(id_to_raw.get(i, i)) for i in range(n_classes)],
})
label_contract.to_csv(DMF_ARTIFACT_DIR / 'label_mapping.csv', index=False, encoding='utf-8-sig')

df['rating_date'] = pd.to_datetime(df['rating_date'], errors='coerce', format='mixed')
if 'sector' not in df.columns:
    df['sector'] = 'UNKNOWN'
df['sector'] = df['sector'].fillna('UNKNOWN').astype(str)
if 'ticker' not in df.columns:
    df['ticker'] = 'UNKNOWN'
df['ticker'] = df['ticker'].fillna('UNKNOWN').astype(str)
if 'company_name' not in df.columns:
    df['company_name'] = df['ticker']
df['company_name'] = df['company_name'].fillna(df['ticker']).astype(str)

sector_encoder = LabelEncoder()
df['sector_id'] = sector_encoder.fit_transform(df['sector'])
n_sectors = int(df['sector_id'].nunique())

train_mask_raw = df['__split__'].eq('train')
stats_ref = df.loc[train_mask_raw].copy()
for c in FINANCIAL_FEATURES:
    med = stats_ref[c].median() if stats_ref[c].notna().any() else 0.0
    df[c] = df[c].fillna(float(0.0 if pd.isna(med) else med))
for c in FINANCIAL_FEATURES:
    lo = stats_ref[c].quantile(0.01)
    hi = stats_ref[c].quantile(0.99)
    if pd.notna(lo) and pd.notna(hi):
        df[c] = df[c].clip(float(lo), float(hi))

df = df.sort_values(['ticker', 'rating_date', '__split__', '__split_row_index__']).reset_index(drop=True)
for c in FINANCIAL_FEATURES:
    df[f'{c}_delta'] = df.groupby('ticker')[c].diff().fillna(0.0)
MODEL_FEATURES = FINANCIAL_FEATURES + [f'{c}_delta' for c in FINANCIAL_FEATURES]

scaler = RobustScaler()
scaler.fit(df.loc[df['__split__'].eq('train'), MODEL_FEATURES].values)
df[MODEL_FEATURES] = scaler.transform(df[MODEL_FEATURES].values)

df['last_y'] = df.groupby('ticker')[TARGET_COL].shift(1)
df['last_y'] = df['last_y'].fillna(df[TARGET_COL]).astype(int)

x_all = torch.tensor(df[MODEL_FEATURES].values.astype(np.float32), dtype=torch.float32, device=device)
y_all = torch.tensor(df[TARGET_COL].values.astype(int), dtype=torch.long, device=device)
last_y_all = torch.tensor(df['last_y'].values.astype(int), dtype=torch.long, device=device)
sector_all = torch.tensor(df['sector_id'].values.astype(int), dtype=torch.long, device=device)

train_mask = torch.tensor(df['__split__'].eq('train').values, dtype=torch.bool, device=device)
val_mask = torch.tensor(df['__split__'].eq('val').values, dtype=torch.bool, device=device)
test_mask = torch.tensor(df['__split__'].eq('test').values, dtype=torch.bool, device=device)

SYNTHETIC_SAMPLE_WEIGHT = 0.55
sample_weight_values = np.ones(len(df), dtype=np.float32)
synthetic_train_count = 0
if 'is_synthetic' in df.columns:
    synthetic_flag = pd.to_numeric(df['is_synthetic'], errors='coerce').fillna(0).astype(int)
    synthetic_train_mask_np = (df['__split__'].eq('train') & synthetic_flag.eq(1)).to_numpy()
    synthetic_train_count = int(synthetic_train_mask_np.sum())
    sample_weight_values[synthetic_train_mask_np] = SYNTHETIC_SAMPLE_WEIGHT
else:
    synthetic_flag = pd.Series(np.zeros(len(df), dtype=int), index=df.index)
sample_weight_all = torch.tensor(sample_weight_values, dtype=torch.float32, device=device)

train_class_counts = torch.bincount(y_all[train_mask], minlength=n_classes).float()
train_effective_class_counts = torch.zeros(n_classes, dtype=torch.float32, device=device)
train_effective_class_counts.scatter_add_(0, y_all[train_mask], sample_weight_all[train_mask])
train_class_weights = train_effective_class_counts.sum() / train_effective_class_counts.clamp_min(1.0)
train_class_weights = train_class_weights / train_class_weights.mean().clamp_min(1e-12)

# CE remains the anchor. Synthetic rows are down-weighted instead of applying
# aggressive class weights, which tends to hurt overall test accuracy here.
class0_mild_weights = torch.ones(n_classes, dtype=torch.float32, device=device)
if n_classes > 0:
    class0_mild_weights[0] = 1.20

print('Train class counts:', train_class_counts.detach().cpu().numpy().astype(int).tolist())
print('Effective weighted train class counts:', np.round(train_effective_class_counts.detach().cpu().numpy(), 2).tolist())
print('Train inverse-frequency weights:', np.round(train_class_weights.detach().cpu().numpy(), 4).tolist())
print('Mild class-0 weights available to final loss:', np.round(class0_mild_weights.detach().cpu().numpy(), 4).tolist())
print('Synthetic train rows:', synthetic_train_count, '| synthetic sample weight:', SYNTHETIC_SAMPLE_WEIGHT)

print('Rows train/val/test:', int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum()))
print('n_classes:', n_classes, '| n_sectors:', n_sectors, '| n_features:', len(MODEL_FEATURES))


In [ ]:
CLASS0_LABEL_ID = 0

CLASS0_THRESHOLD_CONFIG = {
    'enabled': True,
    'metric': 'Class0_F2',
    'accuracy_floor_drop': 0.005,
    'threshold_grid': np.round(np.arange(0.05, 0.501, 0.01), 2).tolist(),
}


def predict_with_class0_threshold(proba, class0_threshold=None):
    pred = np.asarray(proba).argmax(axis=1).astype(int)
    if class0_threshold is None:
        return pred
    promote_mask = np.asarray(proba)[:, CLASS0_LABEL_ID] >= float(class0_threshold)
    pred[promote_mask] = CLASS0_LABEL_ID
    return pred


def compute_metrics(y_true, y_pred, proba, n_cls, last_y=None):
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    acc = accuracy_score(y_true_arr, y_pred_arr)
    f1m = f1_score(y_true_arr, y_pred_arr, average='macro', zero_division=0)
    f1w = f1_score(y_true_arr, y_pred_arr, average='weighted', zero_division=0)
    prec = precision_score(y_true_arr, y_pred_arr, average='weighted', zero_division=0)
    rec = recall_score(y_true_arr, y_pred_arr, average='weighted', zero_division=0)
    class_prec, class_rec, class_f1, class_support = precision_recall_fscore_support(
        y_true_arr,
        y_pred_arr,
        labels=list(range(n_cls)),
        zero_division=0,
    )
    c0_precision = float(class_prec[CLASS0_LABEL_ID]) if CLASS0_LABEL_ID < len(class_prec) else float('nan')
    c0_recall = float(class_rec[CLASS0_LABEL_ID]) if CLASS0_LABEL_ID < len(class_rec) else float('nan')
    c0_f1 = float(class_f1[CLASS0_LABEL_ID]) if CLASS0_LABEL_ID < len(class_f1) else float('nan')
    c0_support = int(class_support[CLASS0_LABEL_ID]) if CLASS0_LABEL_ID < len(class_support) else 0
    if c0_precision + c0_recall > 0:
        c0_f2 = float(5.0 * c0_precision * c0_recall / (4.0 * c0_precision + c0_recall))
    else:
        c0_f2 = 0.0
    qwk = cohen_kappa_score(y_true_arr, y_pred_arr, weights='quadratic')
    try:
        y_bin = label_binarize(y_true_arr, classes=list(range(n_cls)))
        auc_score = roc_auc_score(y_bin, proba, average='macro', multi_class='ovr')
    except Exception:
        auc_score = float('nan')
    ordinal_mae = np.mean(np.abs(y_true_arr - y_pred_arr))
    # ChgAcc: accuracy on samples where label changed vs last known rating.
    if last_y is not None:
        last_y_arr = np.asarray(last_y)
        change_mask = last_y_arr != y_true_arr
        if change_mask.sum() > 0:
            chg_acc = float(accuracy_score(y_true_arr[change_mask], y_pred_arr[change_mask]))
        else:
            chg_acc = float('nan')
    else:
        chg_acc = float('nan')
    return {
        'Accuracy': float(acc),
        'Precision_Weighted': float(prec),
        'Recall_Weighted': float(rec),
        'Macro_F1': float(f1m),
        'Weighted_F1': float(f1w),
        'Class0_Precision': c0_precision,
        'Class0_Recall': c0_recall,
        'Class0_F1': c0_f1,
        'Class0_F2': c0_f2,
        'Class0_Support': c0_support,
        'AUC': float(auc_score),
        'QWK': float(qwk),
        'ChgAcc': chg_acc,
        'Ordinal_MAE': float(ordinal_mae),
    }


def evaluate_logits(logits, mask, class0_threshold=None):
    probs = torch.softmax(logits[mask], dim=1).detach().cpu().numpy()
    y_true = y_all[mask].detach().cpu().numpy()
    y_pred = predict_with_class0_threshold(probs, class0_threshold=class0_threshold)
    last_y_np = last_y_all[mask].detach().cpu().numpy()
    return compute_metrics(y_true, y_pred, probs, n_classes, last_y=last_y_np), y_true, y_pred, probs


def selection_score(metrics):
    chg_acc = 0.0 if np.isnan(metrics['ChgAcc']) else metrics['ChgAcc']
    return (
        0.60 * metrics['Accuracy']
        + 0.15 * metrics['QWK']
        + 0.10 * metrics['Macro_F1']
        + 0.10 * metrics['Class0_F2']
        + 0.05 * chg_acc
        - 0.05 * metrics['Ordinal_MAE']
    )


def calibrate_class0_threshold(y_true, proba, last_y=None, config=None):
    config = config or CLASS0_THRESHOLD_CONFIG
    baseline_pred = predict_with_class0_threshold(proba, class0_threshold=None)
    baseline_metrics = compute_metrics(y_true, baseline_pred, proba, n_classes, last_y=last_y)
    rows = []
    for threshold in config['threshold_grid']:
        pred = predict_with_class0_threshold(proba, class0_threshold=threshold)
        metrics = compute_metrics(y_true, pred, proba, n_classes, last_y=last_y)
        rows.append({'class0_threshold': float(threshold), **metrics})
    sweep_df = pd.DataFrame(rows)
    accuracy_floor = baseline_metrics['Accuracy'] - float(config.get('accuracy_floor_drop', 0.01))
    candidates = sweep_df[sweep_df['Accuracy'] >= accuracy_floor].copy()
    if candidates.empty:
        candidates = sweep_df.copy()
    sort_cols = [config.get('metric', 'Class0_F2'), 'Accuracy', 'Macro_F1', 'QWK']
    best_row = candidates.sort_values(sort_cols, ascending=False).iloc[0]
    best_threshold = float(best_row['class0_threshold'])
    return best_threshold, sweep_df, baseline_metrics, best_row.to_dict()


In [ ]:
# CCR-GNN / Corporation-to-Graph configuration from Feng et al. (2012.01933v1).
# Paper mapping adapted to this notebook:
# - one rating row is one corporate graph;
# - feature nodes are dimensions of [financial features, last-rating embedding, sector embedding];
# - adjacency is built from self-outer-product magnitude and an adaptive threshold;
# - the threshold is lowered per row until the feature graph is connected, following Algorithm 1;
# - GFIL uses 3 GAT layers with mean/mean/max graph pooling.

LAST_Y_EMB_DIM = 4
SECTOR_EMB_DIM = 8
CCR_GNN_LAYER_DIMS = (8, 64, 16)
CCR_GNN_POOLING = ('mean', 'mean', 'max')
C2G_THRESHOLD_QUANTILE = 0.90
C2G_MIN_THRESHOLD_QUANTILE = 0.20
C2G_THRESHOLD_STEPS = 10
C2G_STRONG_NEIGHBORS = 0
C2G_USE_ABS_OUTER_PRODUCT = True
C2G_ADAPTIVE_CONNECTED = True
C2G_USE_FEATURE_CHAIN_FALLBACK = False

FEATURE_NODE_NAMES = (
    list(MODEL_FEATURES)
    + [f'last_y_emb_{i}' for i in range(LAST_Y_EMB_DIM)]
    + [f'sector_emb_{i}' for i in range(SECTOR_EMB_DIM)]
)

# Kept as None so downstream code cannot accidentally reuse the old inter-company graph.
edge_index = None
edge_df = pd.DataFrame(columns=['src', 'dst', 'src_feature', 'dst_feature'])

c2g_contract = {
    'paper': 'CCR-GNN / corporation-to-graph, arXiv:2012.01933v1',
    'row_graph': 'one graph per rating row/company observation',
    'node_count': len(FEATURE_NODE_NAMES),
    'node_sources': {
        'financial_and_delta_features': len(MODEL_FEATURES),
        'last_rating_embedding_dims': LAST_Y_EMB_DIM,
        'sector_embedding_dims': SECTOR_EMB_DIM,
    },
    'adjacency': 'self_outer_product_adaptive_threshold_until_connected + self_loops',
    'threshold_quantile_start': C2G_THRESHOLD_QUANTILE,
    'threshold_quantile_min': C2G_MIN_THRESHOLD_QUANTILE,
    'threshold_steps': C2G_THRESHOLD_STEPS,
    'strong_neighbors': C2G_STRONG_NEIGHBORS,
    'adaptive_connected': C2G_ADAPTIVE_CONNECTED,
    'feature_chain_fallback': C2G_USE_FEATURE_CHAIN_FALLBACK,
    'gat_layer_dims': CCR_GNN_LAYER_DIMS,
    'pooling': CCR_GNN_POOLING,
}
print('CCR-GNN graph contract:', c2g_contract)


In [ ]:
class DenseFeatureGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=1, dropout=0.2, concat=False, negative_slope=0.2):
        super().__init__()
        self.heads = int(heads)
        self.out_dim = int(out_dim)
        self.concat = bool(concat)
        self.lin = nn.Linear(in_dim, out_dim * self.heads, bias=False)
        self.attn_src = nn.Parameter(torch.empty(self.heads, out_dim))
        self.attn_dst = nn.Parameter(torch.empty(self.heads, out_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim * self.heads if self.concat else out_dim))
        self.dropout = nn.Dropout(dropout)
        self.negative_slope = float(negative_slope)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.attn_src)
        nn.init.xavier_uniform_(self.attn_dst)
        nn.init.zeros_(self.bias)

    def forward(self, node_features, adjacency):
        # node_features: [batch, feature_nodes, channels]
        batch_size, n_nodes, _ = node_features.shape
        h = self.lin(node_features).view(batch_size, n_nodes, self.heads, self.out_dim)
        src_scores = (h * self.attn_src.view(1, 1, self.heads, self.out_dim)).sum(-1).permute(0, 2, 1)
        dst_scores = (h * self.attn_dst.view(1, 1, self.heads, self.out_dim)).sum(-1).permute(0, 2, 1)
        scores = dst_scores.unsqueeze(-1) + src_scores.unsqueeze(-2)
        scores = F.leaky_relu(scores, negative_slope=self.negative_slope)
        scores = scores.masked_fill(~adjacency.unsqueeze(1), -1e9)
        alpha = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.einsum('bhij,bjho->biho', alpha, h)
        if self.concat:
            out = out.reshape(batch_size, n_nodes, self.heads * self.out_dim)
        else:
            out = out.mean(dim=2)
        return out + self.bias


class CCRGNN(nn.Module):
    def __init__(
        self,
        n_features,
        n_classes,
        n_sectors,
        layer_dims=(8, 48, 3),
        pooling=('mean', 'mean', 'max'),
        dropout=0.35,
        last_y_emb_dim=4,
        sector_emb_dim=8,
        context_dropout=0.25,
        c2g_threshold_quantile=0.86,
        c2g_min_threshold_quantile=0.20,
        c2g_threshold_steps=10,
        strong_neighbors=0,
        use_abs_outer_product=True,
        adaptive_connected=True,
        use_feature_chain_fallback=False,
    ):
        super().__init__()
        if len(layer_dims) != len(pooling):
            raise ValueError('layer_dims va pooling phai co cung do dai.')
        self.context_dropout = float(context_dropout)
        self.c2g_threshold_quantile = float(c2g_threshold_quantile)
        self.c2g_min_threshold_quantile = float(c2g_min_threshold_quantile)
        self.c2g_threshold_steps = max(1, int(c2g_threshold_steps))
        self.strong_neighbors = max(0, int(strong_neighbors))
        self.use_abs_outer_product = bool(use_abs_outer_product)
        self.adaptive_connected = bool(adaptive_connected)
        self.use_feature_chain_fallback = bool(use_feature_chain_fallback)
        self.n_features = int(n_features)
        self.context_unknown_id = int(n_classes)
        self.last_y_emb_dim = int(last_y_emb_dim)
        self.last_y_emb = nn.Embedding(n_classes + 1, last_y_emb_dim)
        self.sector_emb = nn.Embedding(n_sectors, sector_emb_dim)
        self.node_count = int(n_features + last_y_emb_dim + sector_emb_dim)
        self.input_norm = nn.LayerNorm(self.node_count)
        self.pooling = tuple(pooling)

        dims = [1, *[int(d) for d in layer_dims]]
        self.gat_layers = nn.ModuleList([
            DenseFeatureGATLayer(dims[i], dims[i + 1], heads=1, dropout=dropout, concat=False)
            for i in range(len(layer_dims))
        ])
        self.gat_residuals = nn.ModuleList([
            nn.Identity() if dims[i] == dims[i + 1] else nn.Linear(dims[i], dims[i + 1], bias=False)
            for i in range(len(layer_dims))
        ])
        self.gat_norms = nn.ModuleList([nn.LayerNorm(int(dim)) for dim in layer_dims])
        self.layer_dropout = nn.Dropout(dropout * 0.5)
        readout_dim = self.node_count * (1 + sum(layer_dims)) + sum(layer_dims)
        self.head = nn.Sequential(
            nn.LayerNorm(readout_dim),
            nn.Dropout(dropout),
            nn.Linear(readout_dim, 128),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def build_node_values(self, x, last_y, sector_id):
        last_y_emb = self.last_y_emb(last_y)
        corporate_embedding = torch.cat([x, last_y_emb, self.sector_emb(sector_id)], dim=1)
        corporate_embedding = self.input_norm(corporate_embedding)
        return corporate_embedding.unsqueeze(-1)

    def apply_context_dropout(self, node_values):
        if not self.training or self.context_dropout <= 0:
            return node_values
        values = node_values.squeeze(-1).clone()
        start = self.n_features
        end = start + self.last_y_emb_dim
        values[:, start:end] = F.dropout(values[:, start:end], p=self.context_dropout, training=True)
        return values.unsqueeze(-1)

    @staticmethod
    def connected_mask(adjacency):
        n_nodes = adjacency.size(1)
        reach = adjacency.clone()
        hops = 1
        while hops < n_nodes:
            reach = reach | torch.bmm(reach.float(), reach.float()).gt(0)
            hops *= 2
        return reach[:, 0, :].all(dim=1)

    def threshold_adjacency(self, scores, eye, off_scores, quantile):
        thresholds = torch.quantile(
            off_scores,
            q=float(quantile),
            dim=1,
            keepdim=True,
        ).view(-1, 1, 1)
        adjacency = scores >= thresholds
        adjacency = adjacency | eye.unsqueeze(0)

        n_nodes = scores.size(1)
        if self.strong_neighbors > 0 and n_nodes > 1:
            k = min(self.strong_neighbors, n_nodes - 1)
            masked_scores = scores.masked_fill(eye.unsqueeze(0), float('-inf'))
            top_idx = torch.topk(masked_scores, k=k, dim=-1).indices
            top_adj = torch.zeros_like(adjacency)
            batch_idx = torch.arange(scores.size(0), device=scores.device).view(-1, 1, 1).expand(-1, n_nodes, k)
            row_idx = torch.arange(n_nodes, device=scores.device).view(1, -1, 1).expand(scores.size(0), -1, k)
            top_adj[batch_idx, row_idx, top_idx] = True
            adjacency = adjacency | top_adj | top_adj.transpose(1, 2)
        return adjacency

    def c2g_adjacency(self, node_values):
        # Adaptive C2G mirrors Algorithm 1 in the paper: start sparse, then lower
        # the per-row threshold until each corporation feature graph is connected.
        with torch.no_grad():
            values = node_values.squeeze(-1)
            scores = values.unsqueeze(2) * values.unsqueeze(1)
            if self.use_abs_outer_product:
                scores = scores.abs()
            n_nodes = scores.size(1)
            eye = torch.eye(n_nodes, device=scores.device, dtype=torch.bool)
            off_diag = ~eye
            off_scores = scores[:, off_diag]
            if off_scores.numel() == 0:
                return eye.unsqueeze(0).expand(scores.size(0), -1, -1).clone()

            if not self.adaptive_connected:
                adjacency = self.threshold_adjacency(scores, eye, off_scores, self.c2g_threshold_quantile)
            else:
                quantiles = torch.linspace(
                    self.c2g_threshold_quantile,
                    self.c2g_min_threshold_quantile,
                    steps=self.c2g_threshold_steps,
                    device=scores.device,
                ).tolist()
                quantiles.append(0.0)
                adjacency = None
                remaining = torch.ones(scores.size(0), device=scores.device, dtype=torch.bool)
                for q in quantiles:
                    candidate = self.threshold_adjacency(scores, eye, off_scores, q)
                    connected = self.connected_mask(candidate)
                    if adjacency is None:
                        adjacency = candidate.clone()
                    use_now = remaining & connected
                    if use_now.any():
                        adjacency[use_now] = candidate[use_now]
                    remaining = remaining & ~connected
                    if not remaining.any():
                        break
                if remaining.any():
                    candidate = self.threshold_adjacency(scores, eye, off_scores, 0.0)
                    adjacency[remaining] = candidate[remaining]

            if self.use_feature_chain_fallback and n_nodes > 1:
                chain = torch.zeros((n_nodes, n_nodes), device=scores.device, dtype=torch.bool)
                idx = torch.arange(n_nodes - 1, device=scores.device)
                chain[idx, idx + 1] = True
                chain[idx + 1, idx] = True
                adjacency = adjacency | chain.unsqueeze(0)
        return adjacency

    @staticmethod
    def graph_pool(node_features, mode):
        if mode == 'mean':
            return node_features.mean(dim=1)
        if mode == 'max':
            return node_features.max(dim=1).values
        raise ValueError(f'Pooling khong ho tro: {mode}')

    def forward(self, x, last_y, sector_id, edge_index=None, return_embeddings=False, context_mask_prob=0.0):
        if self.training and context_mask_prob > 0.0:
            mask = torch.rand(last_y.shape, device=last_y.device) < float(context_mask_prob)
            last_y = last_y.masked_fill(mask, self.context_unknown_id)
        node_values = self.build_node_values(x, last_y, sector_id)
        adjacency = self.c2g_adjacency(node_values)
        node_values = self.apply_context_dropout(node_values)
        local_reps = [node_values]
        global_reps = []
        h = node_values
        for layer, residual, norm, pool_mode in zip(self.gat_layers, self.gat_residuals, self.gat_norms, self.pooling):
            h_next = F.elu(layer(h, adjacency))
            h = norm(h_next + residual(h))
            h = self.layer_dropout(h)
            local_reps.append(h)
            global_reps.append(self.graph_pool(h, pool_mode))
        local_readout = torch.cat([rep.reshape(rep.size(0), -1) for rep in local_reps], dim=1)
        global_readout = torch.cat(global_reps, dim=1)
        readout = torch.cat([local_readout, global_readout], dim=1)
        logits = self.head(readout)
        if return_embeddings:
            return logits, readout
        return logits


CreditGAT = CCRGNN


class RampedFocalOrdinalLoss(nn.Module):
    def __init__(
        self,
        n_classes,
        class_weights=None,
        ce_weight=1.0,
        focal_gamma=0.25,
        focal_weight=0.0,
        ordinal_weight=0.0005,
        class0_margin_weight=0.0,
        class0_margin=0.10,
        label_smoothing=0.0,
        warmup_epochs=100,
    ):
        super().__init__()
        self.n_classes = int(n_classes)
        self.ce_weight = float(ce_weight)
        self.focal_gamma = float(focal_gamma)
        self.focal_weight = float(focal_weight)
        self.ordinal_weight = float(ordinal_weight)
        self.class0_margin_weight = float(class0_margin_weight)
        self.class0_margin = float(class0_margin)
        self.label_smoothing = float(label_smoothing)
        self.warmup_epochs = max(1, int(warmup_epochs))
        if class_weights is None:
            self.register_buffer('class_weights', None)
        else:
            self.register_buffer('class_weights', class_weights.detach().float().clone())
        self.register_buffer('class_positions', torch.arange(self.n_classes, dtype=torch.float32))

    @staticmethod
    def weighted_mean(values, sample_weights=None):
        if sample_weights is None:
            return values.mean()
        weights = sample_weights.to(values.device).float().view(-1)
        weights = weights / weights.mean().clamp_min(1e-8)
        return (values * weights).sum() / weights.sum().clamp_min(1e-8)

    def ramp(self, epoch=None):
        if epoch is None:
            return 1.0
        return min(1.0, max(0.0, float(epoch) / float(self.warmup_epochs)))

    def class0_margin_loss(self, logits, targets, sample_weights=None):
        class0_mask = targets.eq(0)
        if class0_mask.sum() == 0 or logits.size(1) <= 1:
            return logits.new_tensor(0.0)
        class0_logits = logits[class0_mask, 0]
        competitor_logits = logits[class0_mask, 1:].max(dim=1).values
        class0_margin = class0_logits - competitor_logits
        margin_terms = F.softplus(self.class0_margin - class0_margin)
        class0_weights = sample_weights[class0_mask] if sample_weights is not None else None
        return self.weighted_mean(margin_terms, class0_weights)

    def loss_parts(self, logits, targets, epoch=None, sample_weights=None):
        targets = targets.long()
        ce_per_sample = F.cross_entropy(
            logits,
            targets,
            weight=self.class_weights,
            reduction='none',
            label_smoothing=self.label_smoothing,
        )
        ce_loss = self.weighted_mean(ce_per_sample, sample_weights)

        probs = torch.softmax(logits, dim=1)
        if self.focal_weight > 0:
            pt = probs.gather(1, targets.view(-1, 1)).squeeze(1).clamp_min(1e-8)
            focal_terms = (1.0 - pt) ** self.focal_gamma * ce_per_sample
            focal_loss = self.weighted_mean(focal_terms, sample_weights)
        else:
            focal_loss = logits.new_tensor(0.0)

        if self.ordinal_weight > 0:
            distances = torch.abs(self.class_positions.to(logits.device).view(1, -1) - targets.float().view(-1, 1))
            ordinal_terms = (probs * distances).sum(dim=1)
            if self.n_classes > 1:
                ordinal_terms = ordinal_terms / float(self.n_classes - 1)
            ordinal_loss = self.weighted_mean(ordinal_terms, sample_weights)
        else:
            ordinal_loss = logits.new_tensor(0.0)

        margin_loss = (
            self.class0_margin_loss(logits, targets, sample_weights=sample_weights)
            if self.class0_margin_weight > 0
            else logits.new_tensor(0.0)
        )
        ramp = self.ramp(epoch)
        aux_loss = ramp * (
            self.focal_weight * focal_loss
            + self.ordinal_weight * ordinal_loss
            + self.class0_margin_weight * margin_loss
        )
        total_loss = self.ce_weight * ce_loss + aux_loss
        return total_loss, {
            'ce_loss': ce_loss,
            'focal_loss': focal_loss,
            'ordinal_loss': ordinal_loss,
            'class0_margin_loss': margin_loss,
            'aux_loss': aux_loss,
        }

    def forward(self, logits, targets, epoch=None, sample_weights=None):
        total_loss, _ = self.loss_parts(logits, targets, epoch=epoch, sample_weights=sample_weights)
        return total_loss


model = CreditGAT(
    n_features=len(MODEL_FEATURES),
    n_classes=n_classes,
    n_sectors=n_sectors,
    layer_dims=CCR_GNN_LAYER_DIMS,
    pooling=CCR_GNN_POOLING,
    dropout=0.30,
    last_y_emb_dim=LAST_Y_EMB_DIM,
    sector_emb_dim=SECTOR_EMB_DIM,
    context_dropout=0.18,
    c2g_threshold_quantile=C2G_THRESHOLD_QUANTILE,
    c2g_min_threshold_quantile=C2G_MIN_THRESHOLD_QUANTILE,
    c2g_threshold_steps=C2G_THRESHOLD_STEPS,
    strong_neighbors=C2G_STRONG_NEIGHBORS,
    use_abs_outer_product=C2G_USE_ABS_OUTER_PRODUCT,
    adaptive_connected=C2G_ADAPTIVE_CONNECTED,
    use_feature_chain_fallback=C2G_USE_FEATURE_CHAIN_FALLBACK,
).to(device)

LOSS_CONFIG = {
    'ce_weight': 1.0,
    'focal_gamma': 0.25,
    'focal_weight': 0.015,
    'ordinal_weight': 0.002,
    'class0_margin_weight': 0.0,
    'class0_margin': 0.10,
    'label_smoothing': 0.0,
    'warmup_epochs': 100,
    'use_mild_class0_weights': False,
    'synthetic_sample_weight': SYNTHETIC_SAMPLE_WEIGHT,
}
criterion = RampedFocalOrdinalLoss(
    n_classes=n_classes,
    class_weights=class0_mild_weights if LOSS_CONFIG['use_mild_class0_weights'] else None,
    ce_weight=LOSS_CONFIG['ce_weight'],
    focal_gamma=LOSS_CONFIG['focal_gamma'],
    focal_weight=LOSS_CONFIG['focal_weight'],
    ordinal_weight=LOSS_CONFIG['ordinal_weight'],
    class0_margin_weight=LOSS_CONFIG['class0_margin_weight'],
    class0_margin=LOSS_CONFIG['class0_margin'],
    label_smoothing=LOSS_CONFIG['label_smoothing'],
    warmup_epochs=LOSS_CONFIG['warmup_epochs'],
).to(device)
OPTIMIZER_CONFIG = {
    'lr': 1.0e-3,
    'weight_decay': 3e-5,
    'plateau_factor': 0.60,
    'plateau_patience': 8,
    'min_lr': 1e-5,
}
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=OPTIMIZER_CONFIG['lr'],
    weight_decay=OPTIMIZER_CONFIG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=OPTIMIZER_CONFIG['plateau_factor'],
    patience=OPTIMIZER_CONFIG['plateau_patience'],
    min_lr=OPTIMIZER_CONFIG['min_lr'],
)


def summarize_c2g_graph(model, sample_size=512):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        idx = torch.where(train_mask)[0][: min(int(train_mask.sum().item()), int(sample_size))]
        node_values = model.build_node_values(x_all[idx], last_y_all[idx], sector_all[idx])
        adjacency = model.c2g_adjacency(node_values)
        n_nodes = adjacency.size(1)
        eye = torch.eye(n_nodes, device=adjacency.device, dtype=torch.bool).unsqueeze(0)
        off_adj = adjacency & ~eye
        avg_degree = off_adj.float().sum(dim=-1).mean().item()
        edge_density = off_adj.float().mean().item()
        connected_rate = model.connected_mask(adjacency).float().mean().item()
    if was_training:
        model.train()
    return {
        'sample_rows': int(idx.numel()),
        'node_count': int(n_nodes),
        'avg_degree': round(float(avg_degree), 4),
        'edge_density': round(float(edge_density), 4),
        'connected_rate': round(float(connected_rate), 4),
        'threshold_quantile_start': float(model.c2g_threshold_quantile),
        'threshold_quantile_min': float(model.c2g_min_threshold_quantile),
        'threshold_steps': int(model.c2g_threshold_steps),
        'adaptive_connected': bool(model.adaptive_connected),
        'strong_neighbors': int(model.strong_neighbors),
    }
print('CCR-GNN config:', c2g_contract)
print('Loss config:', LOSS_CONFIG)
print('Optimizer config:', OPTIMIZER_CONFIG)
print('Initial C2G graph stats:', summarize_c2g_graph(model))
print(model)


In [ ]:
# Visualization: one corporation-to-graph feature graph.
# Yeu cau: chay cell model truoc. Rerun cell nay sau training neu muon xem graph voi embedding da hoc.
if 'model' not in globals() or 'FEATURE_NODE_NAMES' not in globals():
    raise RuntimeError('Khong tim thay model hoac FEATURE_NODE_NAMES. Hay chay cell cau hinh va cell model truoc.')

import networkx as nx
from matplotlib.colors import Normalize, TwoSlopeNorm

sample_candidates = np.flatnonzero(df['__split__'].eq('val').values)
sample_idx = int(sample_candidates[0]) if len(sample_candidates) else 0

model_was_training = model.training
model.eval()
with torch.no_grad():
    sample_nodes = model.build_node_values(
        x_all[sample_idx:sample_idx + 1],
        last_y_all[sample_idx:sample_idx + 1],
        sector_all[sample_idx:sample_idx + 1],
    )
    sample_adj = model.c2g_adjacency(sample_nodes)[0].detach().cpu().numpy().astype(bool)
    sample_values = sample_nodes[0, :, 0].detach().cpu().numpy()
if model_was_training:
    model.train()

edge_rows = []
for src in range(sample_adj.shape[0]):
    for dst in range(src + 1, sample_adj.shape[1]):
        if sample_adj[src, dst] or sample_adj[dst, src]:
            edge_rows.append({
                'src': src,
                'dst': dst,
                'src_feature': FEATURE_NODE_NAMES[src],
                'dst_feature': FEATURE_NODE_NAMES[dst],
                'outer_abs': float(abs(sample_values[src] * sample_values[dst])),
            })
feature_edge_df = pd.DataFrame(edge_rows).sort_values('outer_abs', ascending=False).reset_index(drop=True)
display(feature_edge_df.head(20))

G = nx.Graph()
for idx, name in enumerate(FEATURE_NODE_NAMES):
    G.add_node(idx, label=name, value=float(sample_values[idx]))
for row in edge_rows:
    G.add_edge(int(row['src']), int(row['dst']), weight=float(row['outer_abs']))

fig, ax = plt.subplots(figsize=(12, 9), dpi=160)
pos = nx.spring_layout(G, seed=SEED, k=0.55, iterations=120)
degrees = dict(G.degree())
node_sizes = [140 + 22 * degrees.get(n, 0) for n in G.nodes]
node_values = [G.nodes[n]['value'] for n in G.nodes]
if min(node_values) < 0 < max(node_values):
    color_norm = TwoSlopeNorm(vmin=min(node_values), vcenter=0.0, vmax=max(node_values))
else:
    color_norm = Normalize(vmin=min(node_values), vmax=max(node_values))
node_colors = plt.get_cmap('coolwarm')(color_norm(node_values))
nx.draw_networkx_edges(G, pos, alpha=0.18, width=0.8, edge_color='#777777', ax=ax)
nodes = nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    linewidths=0.7,
    edgecolors='#222222',
    ax=ax,
)
labels = {
    n: (G.nodes[n]['label'] if len(G.nodes[n]['label']) <= 18 else G.nodes[n]['label'][:16] + '..')
    for n in G.nodes
}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
colorbar = plt.cm.ScalarMappable(cmap='coolwarm', norm=color_norm)
colorbar.set_array([])
fig.colorbar(colorbar, ax=ax, shrink=0.72, label='Normalized node value')
ax.set_title(f'CCR-GNN Corporation-to-Graph Feature Graph | row_id={df.loc[sample_idx, "row_id"]}')
ax.axis('off')
fig.tight_layout()
graph_path = ARTIFACT_DIR / 'gat_graph_visualization.png'
fig.savefig(graph_path, dpi=300, bbox_inches='tight')
plt.show()

print('Feature nodes:', len(FEATURE_NODE_NAMES), '| feature edges:', len(feature_edge_df))
print('Saved:', graph_path)


In [ ]:
history = {
    'epoch': [],
    'train_Loss': [], 'val_Loss': [],
    'train_CE_Loss': [], 'val_CE_Loss': [],
    'train_Aux_Loss': [], 'val_Aux_Loss': [],
    'train_Accuracy': [], 'val_Accuracy': [],
    'train_Macro_F1': [], 'val_Macro_F1': [],
    'train_Class0_Precision': [], 'val_Class0_Precision': [],
    'train_Class0_Recall': [], 'val_Class0_Recall': [],
    'train_Class0_F1': [], 'val_Class0_F1': [],
    'train_Class0_F2': [], 'val_Class0_F2': [],
    'train_ChgAcc': [], 'val_ChgAcc': [],
    'train_Ordinal_MAE': [], 'val_Ordinal_MAE': [],
    'train_AUC': [], 'val_AUC': [],
    'train_QWK': [], 'val_QWK': [],
    'Learning_Rate': [],
    'context_mask_prob': [],
    'val_MetricScore': [], 'val_CheckpointScore': [],
}


CONTEXT_MASK_START = 0.35
CONTEXT_MASK_END = 0.10
CONTEXT_MASK_WARMUP_EPOCHS = 45


def scheduled_context_mask(epoch):
    progress = min(1.0, max(0.0, (float(epoch) - 1.0) / float(max(1, CONTEXT_MASK_WARMUP_EPOCHS - 1))))
    return CONTEXT_MASK_START + (CONTEXT_MASK_END - CONTEXT_MASK_START) * progress


CHECKPOINT_CONFIG = {
    'mode': 'pareto_accuracy_loss',
    'accuracy_tolerance': 0.002,
    'val_loss_weight': 0.18,
    'loss_gap_weight': 0.08,
}


def checkpoint_score(metrics, val_loss, train_loss):
    # Validation-only score: accuracy remains primary, loss/gap decide between close models.
    chg_acc = 0.0 if np.isnan(metrics['ChgAcc']) else metrics['ChgAcc']
    loss_gap = max(0.0, float(val_loss) - float(train_loss))
    return (
        0.70 * metrics['Accuracy']
        + 0.12 * metrics['QWK']
        + 0.07 * metrics['Macro_F1']
        + 0.03 * metrics['Class0_F2']
        + 0.04 * chg_acc
        - CHECKPOINT_CONFIG['val_loss_weight'] * float(val_loss)
        - CHECKPOINT_CONFIG['loss_gap_weight'] * loss_gap
        - 0.03 * metrics['Ordinal_MAE']
    )


def clone_state_to_cpu(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def load_state_from_cpu(model, state):
    model.load_state_dict({k: v.to(device) for k, v in state.items()})


best_val_score = -np.inf
patience, no_improve = 100, 0
max_epochs = 100
epoch_states = {}

for epoch in range(1, max_epochs + 1):
    model.train()
    current_context_mask = scheduled_context_mask(epoch)
    optimizer.zero_grad(set_to_none=True)
    logits = model(x_all, last_y_all, sector_all, context_mask_prob=current_context_mask)
    loss = criterion(
        logits[train_mask],
        y_all[train_mask],
        epoch=epoch,
        sample_weights=sample_weight_all[train_mask],
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_eval = model(x_all, last_y_all, sector_all)
        train_total_loss, train_loss_parts = criterion.loss_parts(
            logits_eval[train_mask],
            y_all[train_mask],
            epoch=epoch,
            sample_weights=sample_weight_all[train_mask],
        )
        val_total_loss, val_loss_parts = criterion.loss_parts(logits_eval[val_mask], y_all[val_mask], epoch=epoch)
        train_loss = train_total_loss.item()
        val_loss = val_total_loss.item()
        train_ce_loss = train_loss_parts['ce_loss'].item()
        val_ce_loss = val_loss_parts['ce_loss'].item()
        train_aux_loss = train_loss_parts['aux_loss'].item()
        val_aux_loss = val_loss_parts['aux_loss'].item()
        tr, _, _, _ = evaluate_logits(logits_eval, train_mask)
        va, _, _, _ = evaluate_logits(logits_eval, val_mask)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['epoch'].append(epoch)
    history['train_Loss'].append(float(train_loss))
    history['val_Loss'].append(float(val_loss))
    history['train_CE_Loss'].append(float(train_ce_loss))
    history['val_CE_Loss'].append(float(val_ce_loss))
    history['train_Aux_Loss'].append(float(train_aux_loss))
    history['val_Aux_Loss'].append(float(val_aux_loss))
    history['Learning_Rate'].append(float(current_lr))
    history['context_mask_prob'].append(float(current_context_mask))
    for metric_name in ['Accuracy', 'Macro_F1', 'Class0_Precision', 'Class0_Recall', 'Class0_F1', 'Class0_F2', 'ChgAcc', 'Ordinal_MAE', 'AUC', 'QWK']:
        history[f'train_{metric_name}'].append(float(tr[metric_name]) if not (isinstance(tr[metric_name], float) and tr[metric_name] != tr[metric_name]) else float('nan'))
        history[f'val_{metric_name}'].append(float(va[metric_name]) if not (isinstance(va[metric_name], float) and va[metric_name] != va[metric_name]) else float('nan'))

    val_metric_score = selection_score(va)
    val_selection_score = checkpoint_score(va, val_ce_loss, train_ce_loss)
    history['val_MetricScore'].append(float(val_metric_score))
    history['val_CheckpointScore'].append(float(val_selection_score))
    epoch_states[int(epoch)] = clone_state_to_cpu(model)
    print(
        f"Epoch {epoch:03d} | TrLoss {train_loss:.4f} | VaLoss {val_loss:.4f} | "
        f"TrCE {train_ce_loss:.4f} | VaCE {val_ce_loss:.4f} | "
        f"VaAcc {va['Accuracy']:.4f} | VaF1 {va['Macro_F1']:.4f} | "
        f"VaC0R {va['Class0_Recall']:.4f} | VaC0F2 {va['Class0_F2']:.4f} | "
        f"VaQWK {va['QWK']:.4f} | MetricScore {val_metric_score:.4f} | "
        f"CkptScore {val_selection_score:.4f} | CtxMask {current_context_mask:.2f} | LR {current_lr:.2e}"
    )

    if val_selection_score > best_val_score + 1e-4:
        best_val_score = val_selection_score
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping.')
            break

history_df = pd.DataFrame(history)
loss_for_checkpoint_col = 'val_CE_Loss' if 'val_CE_Loss' in history_df.columns else 'val_Loss'
best_score_idx = history_df['val_CheckpointScore'].idxmax()
best_acc_idx = history_df['val_Accuracy'].idxmax()
best_loss_idx = history_df[loss_for_checkpoint_col].idxmin()
best_acc_value = float(history_df.loc[best_acc_idx, 'val_Accuracy'])
acc_floor = best_acc_value - float(CHECKPOINT_CONFIG.get('accuracy_tolerance', 0.0))
pareto_pool = history_df[history_df['val_Accuracy'] >= acc_floor].copy()
pareto_sort_cols = [loss_for_checkpoint_col, 'val_Ordinal_MAE', 'val_CheckpointScore']
pareto_idx = pareto_pool.sort_values(pareto_sort_cols, ascending=[True, True, False]).index[0]

checkpoint_indices = {
    'best_score': int(best_score_idx),
    'best_acc': int(best_acc_idx),
    'best_loss': int(best_loss_idx),
    'best_pareto': int(pareto_idx),
}
if CHECKPOINT_CONFIG.get('mode') == 'pareto_accuracy_loss':
    selected_checkpoint_tag = 'best_pareto'
else:
    selected_checkpoint_tag = 'best_score'
selected_checkpoint_idx = checkpoint_indices[selected_checkpoint_tag]
selected_checkpoint_epoch = int(history_df.loc[selected_checkpoint_idx, 'epoch'])


def evaluate_checkpoint(tag, row_idx):
    epoch = int(history_df.loc[row_idx, 'epoch'])
    load_state_from_cpu(model, epoch_states[epoch])
    model.eval()
    with torch.no_grad():
        logits_eval = model(x_all, last_y_all, sector_all)
    val_m, _, _, _ = evaluate_logits(logits_eval, val_mask)
    test_m, _, _, _ = evaluate_logits(logits_eval, test_mask)
    base = {
        'Checkpoint': tag,
        'epoch': epoch,
        'val_loss': float(history_df.loc[row_idx, 'val_Loss']),
        'val_ce_loss': float(history_df.loc[row_idx, loss_for_checkpoint_col]),
        'val_checkpoint_score': float(history_df.loc[row_idx, 'val_CheckpointScore']),
        'selected_for_export': tag == selected_checkpoint_tag,
    }
    return [
        {**base, 'Split': 'Val_RawArgmax', **val_m},
        {**base, 'Split': 'Test_RawArgmax', **test_m},
    ]


checkpoint_rows = []
seen_checkpoint_epochs = set()
for tag, row_idx in checkpoint_indices.items():
    epoch = int(history_df.loc[row_idx, 'epoch'])
    unique_tag = tag if epoch not in seen_checkpoint_epochs else f'{tag}_same_epoch'
    seen_checkpoint_epochs.add(epoch)
    checkpoint_rows.extend(evaluate_checkpoint(unique_tag, row_idx))
checkpoint_report = pd.DataFrame(checkpoint_rows)
checkpoint_report_path = ARTIFACT_DIR / 'gat_checkpoint_comparison.csv'
checkpoint_report.to_csv(checkpoint_report_path, index=False, encoding='utf-8-sig')
display(checkpoint_report)

load_state_from_cpu(model, epoch_states[selected_checkpoint_epoch])
model.eval()
with torch.no_grad():
    final_logits, node_embeddings = model(x_all, last_y_all, sector_all, return_embeddings=True)

val_raw_metrics, y_val, y_val_raw_pred, val_proba = evaluate_logits(final_logits, val_mask)
test_raw_metrics, y_test, y_test_raw_pred, test_proba = evaluate_logits(final_logits, test_mask)

val_last_y = last_y_all[val_mask].detach().cpu().numpy()
test_last_y = last_y_all[test_mask].detach().cpu().numpy()

class0_threshold = None
class0_threshold_sweep = pd.DataFrame()
class0_threshold_baseline = val_raw_metrics
class0_threshold_selected = {}
if CLASS0_THRESHOLD_CONFIG['enabled']:
    class0_threshold, class0_threshold_sweep, class0_threshold_baseline, class0_threshold_selected = calibrate_class0_threshold(
        y_val,
        val_proba,
        last_y=val_last_y,
        config=CLASS0_THRESHOLD_CONFIG,
    )

val_metrics, y_val, y_val_pred, val_proba = evaluate_logits(final_logits, val_mask, class0_threshold=class0_threshold)
test_metrics, y_test, y_test_pred, test_proba = evaluate_logits(final_logits, test_mask, class0_threshold=class0_threshold)

history_path = ARTIFACT_DIR / 'gat_training_history.csv'
history_df.to_csv(history_path, index=False, encoding='utf-8-sig')

report = pd.DataFrame([
    {'Checkpoint': selected_checkpoint_tag, 'Checkpoint_Epoch': selected_checkpoint_epoch, 'Split': 'Val_RawArgmax', 'Class0_Threshold': np.nan, **val_raw_metrics},
    {'Checkpoint': selected_checkpoint_tag, 'Checkpoint_Epoch': selected_checkpoint_epoch, 'Split': 'Test_RawArgmax', 'Class0_Threshold': np.nan, **test_raw_metrics},
    {'Checkpoint': selected_checkpoint_tag, 'Checkpoint_Epoch': selected_checkpoint_epoch, 'Split': 'Val_Class0Calibrated', 'Class0_Threshold': class0_threshold, **val_metrics},
    {'Checkpoint': selected_checkpoint_tag, 'Checkpoint_Epoch': selected_checkpoint_epoch, 'Split': 'Test_Class0Calibrated', 'Class0_Threshold': class0_threshold, **test_metrics},
])
display(report)

metrics_path = ARTIFACT_DIR / 'gat_metrics.csv'
report.to_csv(metrics_path, index=False, encoding='utf-8-sig')

threshold_sweep_path = ARTIFACT_DIR / 'gat_class0_threshold_sweep.csv'
class0_threshold_sweep.to_csv(threshold_sweep_path, index=False, encoding='utf-8-sig')

threshold_summary_path = ARTIFACT_DIR / 'gat_class0_threshold_summary.csv'
pd.DataFrame([{
    'selected_threshold': class0_threshold,
    'selection_metric': CLASS0_THRESHOLD_CONFIG['metric'],
    'accuracy_floor_drop': CLASS0_THRESHOLD_CONFIG['accuracy_floor_drop'],
    'selected_checkpoint': selected_checkpoint_tag,
    'selected_checkpoint_epoch': selected_checkpoint_epoch,
    **{f'val_selected_{k}': v for k, v in class0_threshold_selected.items() if isinstance(v, (int, float, np.integer, np.floating))},
}]).to_csv(threshold_summary_path, index=False, encoding='utf-8-sig')

print('Selected checkpoint:', selected_checkpoint_tag, '@ epoch', selected_checkpoint_epoch)
print('Selected class 0 threshold:', class0_threshold)
print('Saved:', metrics_path)
print('Saved:', history_path)
print('Saved:', checkpoint_report_path)
print('Saved:', threshold_sweep_path)
print('Saved:', threshold_summary_path)


In [ ]:
id_to_raw_local = {v: k for k, v in raw_to_id.items()} if 'raw_to_id' in globals() else {i: i for i in range(n_classes)}
class_labels = [str(id_to_raw_local.get(i, i)) for i in range(n_classes)]
label_ids = list(range(n_classes))

cm = confusion_matrix(y_test, y_test_pred, labels=label_ids)
cm_df = pd.DataFrame(cm, index=class_labels, columns=class_labels)

plt.figure(figsize=(6, 5), dpi=160)
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
threshold_title = f' | class0 threshold={class0_threshold:.2f}' if class0_threshold is not None else ''
plt.title(f'GAT Confusion Matrix (Test){threshold_title}')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
cm_plot_path = ARTIFACT_DIR / 'gat_test_confusion_matrix.png'
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.show()

display(cm_df)
print('Classification report (test set, class 0 calibrated):')
print(classification_report(
    y_test,
    y_test_pred,
    labels=label_ids,
    target_names=class_labels,
    digits=4,
    zero_division=0,
))

if 'test_raw_metrics' in globals():
    print('Raw argmax class 0 metrics:')
    print({
        'Class0_Precision': round(test_raw_metrics['Class0_Precision'], 4),
        'Class0_Recall': round(test_raw_metrics['Class0_Recall'], 4),
        'Class0_F1': round(test_raw_metrics['Class0_F1'], 4),
        'Accuracy': round(test_raw_metrics['Accuracy'], 4),
    })
    print('Calibrated class 0 metrics:')
    print({
        'Class0_Precision': round(test_metrics['Class0_Precision'], 4),
        'Class0_Recall': round(test_metrics['Class0_Recall'], 4),
        'Class0_F1': round(test_metrics['Class0_F1'], 4),
        'Accuracy': round(test_metrics['Accuracy'], 4),
    })

cls_report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_test_pred,
        labels=label_ids,
        target_names=class_labels,
        output_dict=True,
        zero_division=0,
    )
).transpose()

cm_csv_path = ARTIFACT_DIR / 'gat_test_confusion_matrix.csv'
cls_csv_path = ARTIFACT_DIR / 'gat_test_classification_report.csv'
cm_df.to_csv(cm_csv_path, encoding='utf-8-sig')
cls_report_df.to_csv(cls_csv_path, encoding='utf-8-sig')

plt.figure(figsize=(10, 8))
y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], test_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'ROC curve of class {class_labels[i]} (area = {roc_auc_val:0.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('GAT - Receiver Operating Characteristic (ROC) - Multiclass')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
roc_plot_path = ARTIFACT_DIR / 'gat_test_roc_curves.png'
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', cm_plot_path)
print('Saved:', cm_csv_path)
print('Saved:', cls_csv_path)
print('Saved:', roc_plot_path)


In [ ]:
# Export prediction CSVs with the same contract as LSTM for DMF/DCS.

def prediction_frame(split_name, split_mask, y_true, proba, pred=None, class0_threshold=None):
    pred = predict_with_class0_threshold(proba, class0_threshold=class0_threshold) if pred is None else np.asarray(pred).astype(int)
    raw_pred = np.argmax(proba, axis=1).astype(int)
    conf = proba[np.arange(len(pred)), pred]
    rows = df.loc[split_mask.detach().cpu().numpy(), ['row_id', 'ticker', 'company_name', 'rating_date']].copy().reset_index(drop=True)
    rows.insert(0, 'split', split_name)
    rows['rating_date'] = pd.to_datetime(rows['rating_date'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('')
    rows['true_label'] = y_true.astype(int)
    rows['true_label_name'] = [str(id_to_raw_local.get(int(y), y)) for y in y_true]
    rows['pred_label'] = pred
    rows['pred_label_name'] = [str(id_to_raw_local.get(int(y), y)) for y in pred]
    rows['raw_pred_label'] = raw_pred
    rows['raw_pred_label_name'] = [str(id_to_raw_local.get(int(y), y)) for y in raw_pred]
    rows['class0_threshold'] = np.nan if class0_threshold is None else float(class0_threshold)
    rows['confidence'] = conf.astype(float)
    rows['raw_confidence'] = np.max(proba, axis=1).astype(float)
    for cls_idx in range(proba.shape[1]):
        rows[f'prob_{cls_idx}'] = proba[:, cls_idx].astype(float)
    return rows


gat_val_predictions = prediction_frame('val', val_mask, y_val, val_proba, pred=y_val_pred, class0_threshold=class0_threshold)
gat_test_predictions = prediction_frame('test', test_mask, y_test, test_proba, pred=y_test_pred, class0_threshold=class0_threshold)

val_csv = DMF_ARTIFACT_DIR / 'gat_val_predictions.csv'
test_csv = DMF_ARTIFACT_DIR / 'gat_test_predictions.csv'
gat_val_predictions.to_csv(val_csv, index=False, encoding='utf-8-sig')
gat_test_predictions.to_csv(test_csv, index=False, encoding='utf-8-sig')

np.save(DMF_ARTIFACT_DIR / 'gat_val_embeddings.npy', node_embeddings[val_mask].detach().cpu().numpy())
np.save(DMF_ARTIFACT_DIR / 'gat_test_embeddings.npy', node_embeddings[test_mask].detach().cpu().numpy())
np.save(ARTIFACT_DIR / 'gat_val_proba.npy', val_proba.astype(np.float32))
np.save(ARTIFACT_DIR / 'gat_test_proba.npy', test_proba.astype(np.float32))
np.save(ARTIFACT_DIR / 'gat_y_val.npy', y_val.astype(int))
np.save(ARTIFACT_DIR / 'gat_y_test.npy', y_test.astype(int))

label_contract.to_csv(DMF_ARTIFACT_DIR / 'label_mapping.csv', index=False, encoding='utf-8-sig')
print(f'[OK] Saved DMF val CSV  -> {val_csv}')
print(f'[OK] Saved DMF test CSV -> {test_csv}')
print(gat_test_predictions.head())


In [ ]:
# Diagnostics: class 0 threshold trade-off and false negatives.
if 'class0_threshold_sweep' not in globals() or class0_threshold_sweep.empty:
    raise RuntimeError('Chua co class0_threshold_sweep. Hay chay lai cell huan luyen truoc.')

display_cols = [
    'class0_threshold', 'Accuracy', 'Macro_F1',
    'Class0_Precision', 'Class0_Recall', 'Class0_F1', 'Class0_F2',
]
threshold_view = class0_threshold_sweep[display_cols].sort_values(
    ['Class0_F2', 'Accuracy', 'Macro_F1'],
    ascending=False,
).head(12)
display(threshold_view)

class0_tradeoff_path = ARTIFACT_DIR / 'gat_class0_threshold_top_candidates.csv'
threshold_view.to_csv(class0_tradeoff_path, index=False, encoding='utf-8-sig')

test_rows = df.loc[test_mask.detach().cpu().numpy(), [
    'row_id', 'ticker', 'company_name', 'rating_date', 'sector',
    TARGET_COL, 'last_y',
]].copy().reset_index(drop=True)
test_rows['pred_label'] = y_test_pred.astype(int)
test_rows['raw_pred_label'] = y_test_raw_pred.astype(int)
for cls_idx in range(test_proba.shape[1]):
    test_rows[f'prob_{cls_idx}'] = test_proba[:, cls_idx]

class0_error_mask = (test_rows[TARGET_COL].astype(int) == CLASS0_LABEL_ID) & (test_rows['pred_label'] != CLASS0_LABEL_ID)
class0_false_negatives = test_rows.loc[class0_error_mask].sort_values('prob_0', ascending=False)
class0_false_negative_path = ARTIFACT_DIR / 'gat_class0_false_negatives.csv'
class0_false_negatives.to_csv(class0_false_negative_path, index=False, encoding='utf-8-sig')

print('Class 0 confusion by last_y:')
display(pd.crosstab(
    test_rows.loc[test_rows[TARGET_COL].astype(int).eq(CLASS0_LABEL_ID), 'last_y'],
    test_rows.loc[test_rows[TARGET_COL].astype(int).eq(CLASS0_LABEL_ID), 'pred_label'],
    rownames=['last_y'],
    colnames=['pred_label'],
))

print('Top class 0 false negatives by prob_0:')
display(class0_false_negatives.head(20))

print('Saved:', class0_tradeoff_path)
print('Saved:', class0_false_negative_path)


In [ ]:
# Visualization: training curves
if 'history_df' not in globals():
    raise RuntimeError('Khong tim thay history_df. Hay chay lai cell huan luyen truoc.')

from matplotlib.ticker import MultipleLocator

sns.set_theme(style='whitegrid', context='paper')
metrics = ['Loss', 'Accuracy', 'Macro_F1', 'Class0_Recall', 'Class0_F2', 'QWK']
loss_train_col = 'train_CE_Loss' if 'train_CE_Loss' in history_df.columns else 'train_Loss'
loss_val_col = 'val_CE_Loss' if 'val_CE_Loss' in history_df.columns else 'val_Loss'
required_cols = [loss_train_col, loss_val_col] + [f'train_{m}' for m in metrics if m != 'Loss'] + [f'val_{m}' for m in metrics if m != 'Loss']
missing = [c for c in required_cols if c not in history_df.columns]
if missing:
    raise RuntimeError(f'Thieu cot trong history_df: {missing}. Hay chay lai cell huan luyen.')

fig, axes = plt.subplots(3, 2, figsize=(12, 10), dpi=160, constrained_layout=True)
axes = axes.ravel()
max_epoch = int(history_df['epoch'].max())

for ax, metric in zip(axes, metrics):
    train_col = loss_train_col if metric == 'Loss' else f'train_{metric}'
    val_col = loss_val_col if metric == 'Loss' else f'val_{metric}'
    ax.plot(history_df['epoch'], history_df[train_col], label='Train', linewidth=1.8, color='#1f77b4')
    ax.plot(history_df['epoch'], history_df[val_col], label='Validation', linewidth=1.8, color='#d62728')
    title = 'CE Loss' if metric == 'Loss' and loss_train_col == 'train_CE_Loss' else metric
    ax.set_title(title, fontsize=11, fontweight='semibold')
    ax.set_xlabel('Epoch')
    ax.set_xlim(0, max_epoch)
    ax.xaxis.set_major_locator(MultipleLocator(10))
    ax.set_ylabel(metric)
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.legend(frameon=True, fontsize=9)

fig.suptitle('GAT Training Curves with Class 0 Monitoring', fontsize=13, fontweight='bold')
curve_path = ARTIFACT_DIR / 'gat_training_curves.png'
fig.savefig(curve_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', curve_path)


In [ ]:
if 'history_df' not in globals():
    raise RuntimeError('Khong tim thay history_df. Hay chay lai cell huan luyen truoc.')

history_df = history_df.copy()
if 'val_MetricScore' not in history_df.columns:
    history_df['val_MetricScore'] = history_df.apply(
        lambda row: selection_score({
            'Accuracy': row['val_Accuracy'],
            'QWK': row['val_QWK'],
            'Macro_F1': row['val_Macro_F1'],
            'Class0_F2': row['val_Class0_F2'],
            'ChgAcc': row['val_ChgAcc'],
            'Ordinal_MAE': row['val_Ordinal_MAE'],
        }),
        axis=1,
    )
if 'val_CheckpointScore' not in history_df.columns:
    history_df['val_CheckpointScore'] = history_df.apply(
        lambda row: checkpoint_score(
            {
                'Accuracy': row['val_Accuracy'],
                'QWK': row['val_QWK'],
                'Macro_F1': row['val_Macro_F1'],
                'Class0_F2': row['val_Class0_F2'],
                'ChgAcc': row['val_ChgAcc'],
                'Ordinal_MAE': row['val_Ordinal_MAE'],
            },
            row['val_CE_Loss'] if 'val_CE_Loss' in row else row['val_Loss'],
            row['train_CE_Loss'] if 'train_CE_Loss' in row else row['train_Loss'],
        ),
        axis=1,
    )

loss_for_min_col = 'val_CE_Loss' if 'val_CE_Loss' in history_df.columns else 'val_Loss'
fallback_selected_idx = history_df['val_CheckpointScore'].idxmax()
summary_selected_idx = selected_checkpoint_idx if 'selected_checkpoint_idx' in globals() else int(fallback_selected_idx)
summary_selected_tag = selected_checkpoint_tag if 'selected_checkpoint_tag' in globals() else 'best_score'
summary_selected_epoch = (
    selected_checkpoint_epoch
    if 'selected_checkpoint_epoch' in globals()
    else int(history_df.loc[summary_selected_idx, 'epoch'])
)
best_score_idx = history_df['val_CheckpointScore'].idxmax()
min_val_loss_idx = history_df[loss_for_min_col].idxmin()
best_val_acc_idx = history_df['val_Accuracy'].idxmax()

row = history_df.loc[summary_selected_idx]
best_score_row = history_df.loc[best_score_idx]
best_loss_row = history_df.loc[min_val_loss_idx]
best_acc_row = history_df.loc[best_val_acc_idx]

print('Selected metrics (validation-only Pareto checkpoint for export):')
print(f'Checkpoint:       {summary_selected_tag} @ epoch {summary_selected_epoch}')
print(f'Train Loss:       {float(row["train_Loss"]):.6f}')
print(f'Val Loss:         {float(row["val_Loss"]):.6f}')
print(f'Train CE Loss:    {float(row["train_CE_Loss"] if "train_CE_Loss" in row else row["train_Loss"]):.6f}')
print(f'Val CE Loss:      {float(row["val_CE_Loss"] if "val_CE_Loss" in row else row["val_Loss"]):.6f}')
print(f'Train Acc:        {float(row["train_Accuracy"]):.6f}')
print(f'Val Acc:          {float(row["val_Accuracy"]):.6f}')
print(f'Val Class0 Recall:{float(row["val_Class0_Recall"]):.6f}')
print(f'Val Class0 F2:    {float(row["val_Class0_F2"]):.6f}')
print(f'Min Val Loss:     {float(best_loss_row[loss_for_min_col]):.6f} @ epoch {int(best_loss_row["epoch"])}')
print(f'Best Val Acc:     {float(best_acc_row["val_Accuracy"]):.6f} @ epoch {int(best_acc_row["epoch"])}')
print(f'Best Score:       {float(best_score_row["val_CheckpointScore"]):.6f} @ epoch {int(best_score_row["epoch"])}')

summary_df = pd.DataFrame([
    {
        'selected_checkpoint': summary_selected_tag,
        'epoch': summary_selected_epoch,
        'train_loss': float(row['train_Loss']),
        'val_loss': float(row['val_Loss']),
        'train_ce_loss': float(row['train_CE_Loss']) if 'train_CE_Loss' in row else float(row['train_Loss']),
        'val_ce_loss': float(row['val_CE_Loss']) if 'val_CE_Loss' in row else float(row['val_Loss']),
        'val_aux_loss': float(row['val_Aux_Loss']) if 'val_Aux_Loss' in row else 0.0,
        'train_acc': float(row['train_Accuracy']),
        'val_acc': float(row['val_Accuracy']),
        'val_class0_recall': float(row['val_Class0_Recall']),
        'val_class0_f2': float(row['val_Class0_F2']),
        'val_metric_score': float(row['val_MetricScore']),
        'val_checkpoint_score': float(row['val_CheckpointScore']),
        'checkpoint_mode': CHECKPOINT_CONFIG.get('mode', 'unknown') if 'CHECKPOINT_CONFIG' in globals() else 'unknown',
        'accuracy_tolerance': CHECKPOINT_CONFIG.get('accuracy_tolerance', np.nan) if 'CHECKPOINT_CONFIG' in globals() else np.nan,
        'min_val_loss_epoch': int(best_loss_row['epoch']),
        'min_val_loss': float(best_loss_row[loss_for_min_col]),
        'min_val_loss_source': loss_for_min_col,
        'best_val_acc_epoch': int(best_acc_row['epoch']),
        'best_val_acc': float(best_acc_row['val_Accuracy']),
        'best_checkpoint_score_epoch': int(best_score_row['epoch']),
        'best_checkpoint_score': float(best_score_row['val_CheckpointScore']),
        'learning_rate': float(row['Learning_Rate']) if 'Learning_Rate' in row else np.nan,
        'selected_class0_threshold': class0_threshold if 'class0_threshold' in globals() else np.nan,
    }
])

training_summary_path = ARTIFACT_DIR / 'gat_training_summary.csv'
summary_df.to_csv(training_summary_path, index=False, encoding='utf-8-sig')
display(summary_df)
print('Saved:', training_summary_path)


## xAI Captum GradientSHAP + LIME Interpretation

Phan xAI duoc trinh bay theo huong paper-ready cho bai toan xep hang tin dung doanh nghiep:

1. **Global Captum GradientSHAP drivers**: cac feature tai chinh/delta nao anh huong lon nhat tren tap test.
2. **Local "Why this class?"**: voi tung doanh nghiep mau, tach feature dang ung ho va chong lai lop duoc giai thich.
3. **Per-class explanation**: giai thich tung lop rating, khong chi lop argmax, de so sanh cac lop gan nhau theo ordinal risk.
4. **LIME local consistency**: LIME dung cung view feature tai chinh voi SHAP, nhung probability mac dinh lay truc tiep tu model/fusion output cua notebook.
5. **Temporal/fusion context**: baseline neural uu tien financial sequence view; ensemble/DMF giai thich tren probability inputs vi day la input that cua tang ket hop.

Luu y: GradientSHAP/LIME giai thich hanh vi cua model, khong duoc dien giai nhu quan he nhan qua tai chinh.


In [ ]:
# ============================================================
# xAI Captum GradientSHAP + LIME aligned with Transformer-LSTM presentation
# Main artifacts:
#   {model_key}_financial_shap_importance_by_class.csv
#   {model_key}_financial_shap_global_importance.csv
#   {model_key}_financial_shap_local_decisions.csv
#   {model_key}_financial_lime_local_decisions.csv
# ============================================================
SHAP_FINANCIAL_ENABLED = True
LIME_FINANCIAL_ENABLED = True
XAI_MODEL_KEY = "gat"
XAI_MODEL_LABEL = "GAT"
XAI_RANDOM_STATE = SEED if "SEED" in globals() else 42
XAI_LOCAL_SAMPLE_COUNT = 4
XAI_LOCAL_TOP_FEATURES = 10
XAI_FINANCIAL_REDUCER = "mean"  # {"mean", "last"}; used when reducing sequence windows.
XAI_EXPLAIN_CLASS_IDS = None  # None = explain all classes; set e.g. [0, 2] to focus.
XAI_ROW_IDS = None  # Optional explicit row_id list for local case studies.
XAI_BACKGROUND_SIZE = 80
SHAP_MAX_SAMPLES = 12
GRADIENT_SHAP_N_SAMPLES = 64
GRADIENT_SHAP_STDEV = 0.05
XAI_PROXY_MAX_ROWS = 2500
XAI_PROXY_EPOCHS = 220
LIME_NUM_SAMPLES = 600
LIME_USE_DIRECT_MODEL_PROBA = True
XAI_BEESWARM_TOP_FEATURES = 15
XAI_LIME_PLOT_TOP_RULES = 15
XAI_LIME_INSTANCE_TOP_RULES = 8

import math
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
except Exception as exc:
    raise RuntimeError("xAI GradientSHAP requires PyTorch for the differentiable attribution head.") from exc


def _xai_artifact_dir():
    path = globals().get("ARTIFACT_DIR", globals().get("DMF_ARTIFACT_DIR", None))
    if path is None:
        path = Path("/kaggle/working/credit_rating_artifacts") if globals().get("IN_KAGGLE", False) else Path("credit_rating_artifacts")
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def _xai_normalize_proba(pred):
    pred = np.asarray(pred, dtype=np.float64)
    if pred.ndim == 1:
        pred = pred.reshape(1, -1)
    pred = np.clip(pred, 1e-9, 1.0)
    return pred / pred.sum(axis=1, keepdims=True)


def _xai_class_names(n_cls):
    if "class_names" in globals() and len(class_names) > 0:
        names = [str(x) for x in class_names]
        return names[:n_cls] if len(names) >= n_cls else names + [str(i) for i in range(len(names), n_cls)]
    if "id_to_name" in globals():
        return [str(id_to_name.get(i, i)) for i in range(n_cls)]
    if "id_to_raw" in globals():
        return [str(id_to_raw.get(i, i)) for i in range(n_cls)]
    if "raw_to_id" in globals():
        inv = {int(v): k for k, v in raw_to_id.items()}
        return [str(inv.get(i, i)) for i in range(n_cls)]
    if "le" in globals() and hasattr(le, "classes_"):
        names = [str(x) for x in le.classes_]
        return names[:n_cls] if len(names) >= n_cls else names + [str(i) for i in range(len(names), n_cls)]
    return [str(i) for i in range(n_cls)]


def _xai_resolve_class_ids(class_ids, n_cls):
    if class_ids is None:
        return list(range(n_cls))
    out = []
    for class_id in class_ids:
        class_id = int(class_id)
        if class_id < 0 or class_id >= n_cls:
            raise ValueError(f"Invalid xAI class id {class_id}; expected 0..{n_cls - 1}.")
        if class_id not in out:
            out.append(class_id)
    if not out:
        raise ValueError("XAI_EXPLAIN_CLASS_IDS resolved to an empty class list.")
    return out


def _xai_prob_columns(frame, prefix=None):
    if prefix is None:
        cols = [c for c in frame.columns if str(c).startswith("prob_")]
    else:
        cols = [c for c in frame.columns if str(c).startswith(f"{prefix}_prob_")]

    def suffix_num(name):
        try:
            return int(str(name).split("_")[-1])
        except Exception:
            return 10**9
    return sorted(cols, key=suffix_num)


def _xai_get_proba():
    for name in ["dmf_probs", "fp", "test_proba", "test_probs", "weighted_probs", "soft_probs", "y_test_proba"]:
        if name in globals():
            arr = np.asarray(globals()[name])
            if arr.ndim == 2 and arr.shape[0] > 0:
                return _xai_normalize_proba(arr), name
    if "model" in globals() and "X_test" in globals() and hasattr(model, "predict_proba"):
        X_arr = X_test.values if isinstance(X_test, pd.DataFrame) else np.asarray(X_test)
        return _xai_normalize_proba(model.predict_proba(X_arr)), "model.predict_proba(X_test)"
    for frame_name in ["test_predictions", "test_results", "test_result_df", "test_results_df", "out"]:
        if frame_name in globals() and isinstance(globals()[frame_name], pd.DataFrame):
            frame = globals()[frame_name]
            cols = _xai_prob_columns(frame)
            if cols:
                return _xai_normalize_proba(frame[cols].to_numpy(dtype=float)), frame_name
    raise RuntimeError("Kh?ng t?m th?y probability matrix cho xAI. H?y ch?y cell inference/evaluation tr??c.")


_XAI_CANONICAL_FINANCIAL_FEATURES = [
    "current_ratio",
    "debt_equity_ratio",
    "gross_profit_margin",
    "operating_profit_margin",
    "ebit_margin",
    "pretax_profit_margin",
    "net_profit_margin",
    "asset_turnover",
    "roe",
    "roa",
    "operating_cashflow_ps",
    "free_cashflow_ps",
]


def _xai_original_financial_feature_names():
    out = []
    for source_name in ["FINANCIAL_FEATURES", "MODEL_FEATURES", "feature_cols", "financial_cols"]:
        values = globals().get(source_name, [])
        if values is None:
            continue
        try:
            names = [str(v) for v in values]
        except TypeError:
            continue
        for col in _XAI_CANONICAL_FINANCIAL_FEATURES:
            if col in names and col not in out:
                out.append(col)
    return out if out else list(_XAI_CANONICAL_FINANCIAL_FEATURES)


def _xai_financial_indices_from_names(names):
    names = [str(c) for c in names]
    wanted = _xai_original_financial_feature_names()
    cols = [c for c in wanted if c in names]
    idx = [names.index(c) for c in cols]
    return idx, cols


def _xai_frame_financial_values(frame):
    cols = [c for c in _xai_original_financial_feature_names() if c in frame.columns]
    if cols:
        return frame[cols].to_numpy(dtype=np.float64), [str(c) for c in cols]
    return None, []


def _xai_reduce_sequence_to_original_financial(seq, candidate_names):
    seq = np.asarray(seq, dtype=np.float64)
    idx, cols = _xai_financial_indices_from_names(candidate_names)
    if idx:
        return seq[:, :, idx].mean(axis=1), cols
    return seq.mean(axis=1), [str(c) for c in candidate_names]


def _xai_get_feature_view(proba):
    if "test_ds" in globals() and hasattr(test_ds, "samples") and len(test_ds.samples) > 0:
        seq = np.stack([s[0] for s in test_ds.samples], axis=0).astype(np.float64)
        candidate_names = globals().get("MODEL_FEATURES", globals().get("FINANCIAL_FEATURES", [f"feature_{i}" for i in range(seq.shape[-1])]))
        X_fin, financial_cols = _xai_reduce_sequence_to_original_financial(seq, candidate_names)
        return X_fin, financial_cols, "original_financial_sequence_mean"
    if "test_X_all" in globals():
        arr = np.asarray(test_X_all, dtype=np.float64)
        if arr.ndim == 3:
            candidate_names = globals().get("MODEL_FEATURES", globals().get("FINANCIAL_FEATURES", [f"feature_{i}" for i in range(arr.shape[-1])]))
            X_fin, financial_cols = _xai_reduce_sequence_to_original_financial(arr, candidate_names)
            return X_fin, financial_cols, "original_financial_sequence_mean"
        if arr.ndim == 2:
            candidate_names = globals().get("flat_feature_names", [f"feature_{i}" for i in range(arr.shape[1])])
            idx, financial_cols = _xai_financial_indices_from_names(candidate_names)
            if idx:
                return arr[:, idx], financial_cols, "original_financial_flat_features"
            if "MODEL_FEATURES" in globals() and arr.shape[1] == len(MODEL_FEATURES):
                idx, financial_cols = _xai_financial_indices_from_names(MODEL_FEATURES)
                if idx:
                    return arr[:, idx], financial_cols, "original_financial_tabular_features"
            return arr, [str(c) for c in candidate_names], "flattened_sequence"
    if "X_test" in globals():
        if isinstance(X_test, pd.DataFrame):
            X_fin, financial_cols = _xai_frame_financial_values(X_test)
            if financial_cols:
                return X_fin, financial_cols, "original_financial_tabular_features"
            arr = X_test.values
        else:
            arr = np.asarray(X_test)
        arr = np.asarray(arr, dtype=np.float64)
        if "MODEL_FEATURES" in globals() and arr.ndim == 2 and arr.shape[1] == len(MODEL_FEATURES):
            idx, financial_cols = _xai_financial_indices_from_names(MODEL_FEATURES)
            if idx:
                return arr[:, idx], financial_cols, "original_financial_tabular_features"
        if "MODEL_FEATURES" in globals() and "INPUT_SIZE" in globals() and arr.ndim == 2 and arr.shape[1] >= int(INPUT_SIZE) * len(MODEL_FEATURES):
            n_base = len(MODEL_FEATURES)
            window = arr[:, :int(INPUT_SIZE) * n_base].reshape(arr.shape[0], int(INPUT_SIZE), n_base)
            idx, financial_cols = _xai_financial_indices_from_names(MODEL_FEATURES)
            if idx:
                return window[:, :, idx].mean(axis=1), financial_cols, "original_financial_tabular_window_mean"
        cols = list(X_test.columns) if isinstance(X_test, pd.DataFrame) else [f"feature_{i}" for i in range(arr.shape[1])]
        idx, financial_cols = _xai_financial_indices_from_names(cols)
        if idx:
            return arr[:, idx], financial_cols, "original_financial_tabular_features"
        return arr, [str(c) for c in cols], "tabular_features"
    if "df" in globals() and "test_mask" in globals():
        mask = test_mask.detach().cpu().numpy() if hasattr(test_mask, "detach") else np.asarray(test_mask)
        frame = df.loc[mask].reset_index(drop=True)
        X_fin, financial_cols = _xai_frame_financial_values(frame)
        if financial_cols:
            return X_fin, financial_cols, "df_test_mask_original_financial_features"
    for frame_name in ["test_df", "df_test", "test_rows", "out"]:
        if frame_name in globals() and isinstance(globals()[frame_name], pd.DataFrame):
            frame = globals()[frame_name]
            X_fin, financial_cols = _xai_frame_financial_values(frame)
            if financial_cols:
                return X_fin, financial_cols, f"{frame_name}_original_financial_features"
            prob_cols = []
            for prefix in ["tlstm", "gat", "lstm", "tcn", "patchtst", "xgboost", "lightgbm", "dmf"]:
                prob_cols.extend(_xai_prob_columns(frame, prefix=prefix))
            if prob_cols:
                return frame[prob_cols].to_numpy(dtype=np.float64), [str(c) for c in prob_cols], f"{frame_name}_probability_inputs"
    if "test_probas" in globals() and isinstance(test_probas, list) and len(test_probas) > 0:
        mats = [np.asarray(p, dtype=np.float64) for p in test_probas]
        if all(m.ndim == 2 for m in mats):
            names = globals().get("MODEL_NAMES", [f"model_{i}" for i in range(len(mats))])
            feature_cols = []
            for model_name, mat in zip(names, mats):
                for class_idx in range(mat.shape[1]):
                    feature_cols.append(f"{model_name}_prob_{class_idx}")
            return np.concatenate(mats, axis=1), feature_cols, "ensemble_probability_inputs"
    return np.asarray(proba, dtype=np.float64), [f"predicted_prob_{i}" for i in range(proba.shape[1])], "model_probability_vector"


def _xai_metadata(n_rows):
    meta_cols = ["row_id", "ticker", "company_name", "rating_date", "sector", "true_label", "true_label_name"]
    for frame_name in ["test_df", "df_test", "test_rows", "out", "test_predictions", "test_results_df"]:
        if frame_name in globals() and isinstance(globals()[frame_name], pd.DataFrame) and len(globals()[frame_name]) >= n_rows:
            frame = globals()[frame_name].reset_index(drop=True)
            cols = [c for c in meta_cols if c in frame.columns]
            if cols:
                return frame.loc[:n_rows - 1, cols].copy()
    if "df" in globals() and "test_mask" in globals():
        mask = test_mask.detach().cpu().numpy() if hasattr(test_mask, "detach") else np.asarray(test_mask)
        frame = df.loc[mask].reset_index(drop=True)
        cols = [c for c in meta_cols if c in frame.columns]
        if cols and len(frame) >= n_rows:
            return frame.loc[:n_rows - 1, cols].copy()
    meta = pd.DataFrame({"test_index": np.arange(n_rows, dtype=int)})
    if "y_test" in globals() and len(y_test) >= n_rows:
        meta["true_label"] = np.asarray(y_test[:n_rows], dtype=int)
    elif "y_true" in globals() and len(y_true) >= n_rows:
        meta["true_label"] = np.asarray(y_true[:n_rows], dtype=int)
    return meta


def _xai_select_indices(meta, proba):
    n_rows = len(meta)
    if XAI_ROW_IDS is not None and "row_id" in meta.columns:
        wanted = {str(x) for x in XAI_ROW_IDS}
        idx = [int(i) for i, row_id in enumerate(meta["row_id"].astype(str)) if row_id in wanted]
        if idx:
            return idx[:XAI_LOCAL_SAMPLE_COUNT]
    confidence = proba.max(axis=1)
    ranked = np.argsort(-confidence)
    selected = []
    preds = proba.argmax(axis=1)
    for class_id in np.unique(preds[ranked]):
        class_ranked = [int(i) for i in ranked if preds[i] == class_id]
        if class_ranked:
            selected.append(class_ranked[0])
        if len(selected) >= XAI_LOCAL_SAMPLE_COUNT:
            break
    for i in ranked:
        i = int(i)
        if i not in selected:
            selected.append(i)
        if len(selected) >= XAI_LOCAL_SAMPLE_COUNT:
            break
    return selected


class _XAIProbabilityHead(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        hidden = int(min(96, max(16, 2 * n_features)))
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return torch.softmax(self.net(x.float()), dim=1)


def _xai_train_probability_head(X_view, proba):
    rng = np.random.default_rng(XAI_RANDOM_STATE)
    X_view = np.asarray(X_view, dtype=np.float64)
    proba = _xai_normalize_proba(proba)
    center = np.nanmedian(X_view, axis=0)
    scale = np.nanstd(X_view, axis=0)
    scale = np.where(scale < 1e-8, 1.0, scale)
    X_std = np.nan_to_num((X_view - center) / scale, nan=0.0, posinf=0.0, neginf=0.0)
    n_rows = len(X_std)
    fit_idx = np.arange(n_rows)
    if n_rows > XAI_PROXY_MAX_ROWS:
        fit_idx = rng.choice(n_rows, size=XAI_PROXY_MAX_ROWS, replace=False)
    device_xai = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    head = _XAIProbabilityHead(X_std.shape[1], proba.shape[1]).to(device_xai)
    x_train = torch.tensor(X_std[fit_idx], dtype=torch.float32, device=device_xai)
    y_train = torch.tensor(proba[fit_idx], dtype=torch.float32, device=device_xai)
    opt = torch.optim.AdamW(head.parameters(), lr=0.01, weight_decay=1e-4)
    head.train()
    for _ in range(int(XAI_PROXY_EPOCHS)):
        opt.zero_grad(set_to_none=True)
        pred = head(x_train)
        loss = torch.mean((pred - y_train) ** 2)
        loss.backward()
        opt.step()
    head.eval()
    with torch.no_grad():
        pred_all = head(torch.tensor(X_std, dtype=torch.float32, device=device_xai)).detach().cpu().numpy()
    fidelity = {
        "proxy_mse": float(np.mean((pred_all - proba) ** 2)),
        "proxy_argmax_agreement": float(np.mean(pred_all.argmax(axis=1) == proba.argmax(axis=1))),
    }
    return head, X_std.astype(np.float32), center, scale, device_xai, fidelity


def _xai_predict_from_head(head, center, scale, device_xai, x_batch):
    x_batch = np.asarray(x_batch, dtype=np.float64)
    if x_batch.ndim == 1:
        x_batch = x_batch.reshape(1, -1)
    x_std = np.nan_to_num((x_batch - center) / scale, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    with torch.no_grad():
        pred = head(torch.tensor(x_std, dtype=torch.float32, device=device_xai)).detach().cpu().numpy()
    return _xai_normalize_proba(pred)


def _xai_gradientshap_attributions(head, X_std, sample_indices, background_indices, class_ids, device_xai):
    x_samples = torch.tensor(X_std[sample_indices], dtype=torch.float32, device=device_xai)
    baselines = torch.tensor(X_std[background_indices], dtype=torch.float32, device=device_xai)
    method = "captum_gradientshap_probability_head"
    attributions = {}
    try:
        from captum.attr import GradientShap
        gs = GradientShap(head)
        for class_id in class_ids:
            attr = gs.attribute(
                x_samples,
                baselines=baselines,
                target=int(class_id),
                n_samples=int(GRADIENT_SHAP_N_SAMPLES),
                stdevs=float(GRADIENT_SHAP_STDEV),
            )
            attributions[int(class_id)] = attr.detach().cpu().numpy()
    except Exception as exc:
        method = f"expected_gradients_fallback: {type(exc).__name__}"
        rng = np.random.default_rng(XAI_RANDOM_STATE)
        for class_id in class_ids:
            rows = []
            for x_np in X_std[sample_indices]:
                accum = []
                for _ in range(int(GRADIENT_SHAP_N_SAMPLES)):
                    b_np = X_std[int(rng.choice(background_indices))]
                    alpha = float(rng.uniform(0.0, 1.0))
                    noise = rng.normal(0.0, float(GRADIENT_SHAP_STDEV), size=x_np.shape).astype(np.float32)
                    interp_np = b_np + alpha * (x_np - b_np) + noise
                    interp = torch.tensor(interp_np.reshape(1, -1), dtype=torch.float32, device=device_xai, requires_grad=True)
                    score = head(interp)[:, int(class_id)].sum()
                    grad = torch.autograd.grad(score, interp)[0].detach().cpu().numpy()[0]
                    accum.append((x_np - b_np) * grad)
                rows.append(np.mean(accum, axis=0))
            attributions[int(class_id)] = np.asarray(rows, dtype=np.float32)
    return attributions, method


def _xai_lime_fallback(x_row, predict_fn, class_id, feature_names, num_features):
    rng = np.random.default_rng(XAI_RANDOM_STATE + int(class_id))
    x_row = np.asarray(x_row, dtype=np.float64)
    perturb = rng.normal(loc=x_row, scale=np.maximum(np.nanstd(x_row), 1e-3), size=(int(LIME_NUM_SAMPLES), len(x_row)))
    perturb[0] = x_row
    probs = _xai_normalize_proba(predict_fn(perturb))[:, int(class_id)]
    distances = np.linalg.norm(perturb - x_row.reshape(1, -1), axis=1)
    kernel_width = math.sqrt(len(x_row)) * 0.75
    weights = np.exp(-(distances ** 2) / max(kernel_width ** 2, 1e-9))
    X_aug = np.column_stack([np.ones(len(perturb)), perturb - x_row.reshape(1, -1)])
    W = np.sqrt(weights).reshape(-1, 1)
    coef = np.linalg.lstsq(X_aug * W, probs * W.ravel(), rcond=None)[0][1:]
    order = np.argsort(-np.abs(coef))[:num_features]
    return [(str(feature_names[j]), float(coef[j])) for j in order]



def _xai_short_text(text, max_len=34):
    text = str(text)
    return text if len(text) <= int(max_len) else text[: max(1, int(max_len) - 3)] + "..."


def _xai_wrap_text(text, width=28, max_lines=3):
    import textwrap

    lines = textwrap.wrap(str(text), width=int(width), break_long_words=False, replace_whitespace=False)
    if not lines:
        return ""
    if len(lines) > int(max_lines):
        lines = lines[: int(max_lines)]
        lines[-1] = _xai_short_text(lines[-1], max(4, int(width) - 1))
    return "\n".join(lines)


def _xai_lime_rule_feature(rule, feature_names):
    rule_text = str(rule)
    for feature in sorted([str(f) for f in feature_names], key=len, reverse=True):
        if feature in rule_text:
            return feature
    return rule_text

def _xai_save_lime_multiclass_instance_plot(row_meta, exp, model_proba, lime_proba, x_row, html_path, lime_available):
    from matplotlib.patches import Rectangle

    row_id = str(row_meta.get("row_id", row_meta.get("test_index", "unknown")))
    safe_row_id = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in row_id)
    class_ids = list(explain_class_ids)
    if lime_available and exp is not None:
        class_items = {
            int(class_id): exp.as_list(label=int(class_id))[: min(int(XAI_LIME_INSTANCE_TOP_RULES), int(XAI_LOCAL_TOP_FEATURES))]
            for class_id in class_ids
        }
    else:
        class_items = {
            int(class_id): _xai_lime_fallback(x_row, _xai_probability_fn, int(class_id), feature_cols, min(int(XAI_LIME_INSTANCE_TOP_RULES), int(XAI_LOCAL_TOP_FEATURES)))
            for class_id in class_ids
        }
    if not any(class_items.values()):
        return None

    support_color = "#ff7f0e"
    oppose_color = "#5f9ea0"
    pred_class = int(np.argmax(model_proba))
    all_weights = [abs(float(weight)) for items in class_items.values() for _, weight in items]
    common_max_abs = max(max(all_weights, default=1e-3), 1e-3)

    max_items = max(len(items) for items in class_items.values())
    fig_height = max(9.6, 1.18 * max_items + 4.2)
    fig_width = max(24.0, 10.5 + 6.6 * len(class_ids))
    fig = plt.figure(figsize=(fig_width, fig_height), constrained_layout=False)
    gs = fig.add_gridspec(1, 2 + len(class_ids), width_ratios=[1.75] + [3.35] * len(class_ids) + [2.65], wspace=0.90)

    ax_prob = fig.add_subplot(gs[0, 0])
    prob_y = np.arange(len(class_names_xai))
    prob_colors = [support_color if i == pred_class else "white" for i in range(len(class_names_xai))]
    ax_prob.barh(prob_y, lime_proba, color=prob_colors, edgecolor="black", height=0.72)
    ax_prob.set_yticks(prob_y)
    ax_prob.set_yticklabels([_xai_wrap_text(name, 16, 2) for name in class_names_xai], fontsize=12)
    ax_prob.set_xlim(0, 1.0)
    ax_prob.set_xticks([])
    ax_prob.set_title("Prediction probabilities", fontsize=16, loc="left")
    for y, value in zip(prob_y, lime_proba):
        ax_prob.text(min(float(value) + 0.03, 0.98), y, f"{float(value):.2f}", va="center", fontsize=12)
    for spine in ax_prob.spines.values():
        spine.set_visible(False)
    ax_prob.tick_params(axis="y", length=0)
    ax_prob.invert_yaxis()

    for panel_idx, class_id in enumerate(class_ids, start=1):
        items = class_items[int(class_id)]
        ax_rules = fig.add_subplot(gs[0, panel_idx])
        if not items:
            ax_rules.axis("off")
            continue
        rule_labels = [_xai_wrap_text(rule, 28, 4) for rule, _ in items]
        weights = np.array([float(weight) for _, weight in items], dtype=float)
        y_pos = np.arange(len(items))
        bar_colors = [support_color if weight >= 0 else oppose_color for weight in weights]
        ax_rules.barh(y_pos, weights, color=bar_colors, height=0.68)
        ax_rules.axvline(0, color="black", linewidth=1.0)
        ax_rules.set_yticks(y_pos)
        ax_rules.set_yticklabels(rule_labels, fontsize=11.2, linespacing=1.15)
        ax_rules.set_xlim(-common_max_abs * 1.32, common_max_abs * 1.32)
        ax_rules.set_xticks([])
        title_color = support_color if int(class_id) == pred_class else oppose_color
        ax_rules.set_title(_xai_wrap_text(class_names_xai[int(class_id)], 16, 2), fontsize=15, color=title_color, pad=20)
        for y, weight in zip(y_pos, weights):
            x_text = weight + (0.025 * common_max_abs if weight >= 0 else -0.025 * common_max_abs)
            ha = "left" if weight >= 0 else "right"
            ax_rules.text(x_text, y, f"{weight:.2f}", va="center", ha=ha, fontsize=10.8)
        for spine in ax_rules.spines.values():
            spine.set_visible(False)
        ax_rules.tick_params(axis="y", length=0)
        ax_rules.invert_yaxis()

    ax_table = fig.add_subplot(gs[0, -1])
    ax_table.axis("off")
    ax_table.text(0.05, 0.98, "Feature", fontsize=15, fontweight="bold", va="top")
    ax_table.text(0.97, 0.98, "Value", fontsize=15, fontweight="bold", ha="right", va="top")
    seen_features = []
    feature_rows = []
    for _, items in class_items.items():
        for rule, weight in items:
            feat = _xai_lime_rule_feature(rule, feature_cols)
            if feat in seen_features:
                continue
            seen_features.append(feat)
            feat_value = np.nan
            if feat in feature_cols:
                feat_value = float(x_row[feature_cols.index(feat)])
            feature_rows.append((feat, feat_value, float(weight)))
            if len(feature_rows) >= int(XAI_LIME_INSTANCE_TOP_RULES):
                break
        if len(feature_rows) >= int(XAI_LIME_INSTANCE_TOP_RULES):
            break
    row_height = 0.78 / max(len(feature_rows), 1)
    for offset, (feat, feat_value, weight) in enumerate(feature_rows):
        y0 = 0.88 - (offset + 1) * row_height
        color = support_color if weight >= 0 else oppose_color
        ax_table.add_patch(Rectangle((0.02, y0), 0.96, row_height * 0.94, transform=ax_table.transAxes, facecolor=color, edgecolor="none", alpha=0.95))
        value_text = "" if not np.isfinite(feat_value) else f"{feat_value:.2f}"
        ax_table.text(0.05, y0 + row_height * 0.47, _xai_wrap_text(feat, 15, 3), transform=ax_table.transAxes, va="center", fontsize=10.8, color="white", fontweight="bold", linespacing=1.08)
        ax_table.text(0.97, y0 + row_height * 0.47, value_text, transform=ax_table.transAxes, va="center", ha="right", fontsize=11.0, color="white")

    title = f"LIME row_id={row_id} | pred={class_names_xai[pred_class]} | all {len(class_ids)} classes"
    fig.suptitle(title, y=0.995, fontsize=15)
    fig.subplots_adjust(left=0.055, right=0.985, top=0.86, bottom=0.08)
    plot_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_lime_row_{safe_row_id}_all_classes_plot.png"
    plt.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    return plot_path


proba, proba_source = _xai_get_proba()
X_view, feature_cols, feature_source = _xai_get_feature_view(proba)
X_view = np.asarray(X_view, dtype=np.float64)
n_rows = min(len(X_view), len(proba))
X_view = X_view[:n_rows]
proba = proba[:n_rows]
class_names_xai = _xai_class_names(proba.shape[1])
explain_class_ids = _xai_resolve_class_ids(XAI_EXPLAIN_CLASS_IDS, proba.shape[1])
meta = _xai_metadata(n_rows).reset_index(drop=True)
selected_indices = _xai_select_indices(meta, proba)
background_size = min(int(XAI_BACKGROUND_SIZE), n_rows)
rng = np.random.default_rng(XAI_RANDOM_STATE)
background_indices = rng.choice(n_rows, size=background_size, replace=False) if n_rows > background_size else np.arange(n_rows)
grad_sample_size = min(int(SHAP_MAX_SAMPLES), n_rows)
grad_sample_indices = rng.choice(n_rows, size=grad_sample_size, replace=False) if n_rows > grad_sample_size else np.arange(n_rows)

print(f"[INFO] xAI for {XAI_MODEL_LABEL}: rows={n_rows}, feature_source={feature_source}, probability_source={proba_source}")
print("[INFO] GradientSHAP and LIME use the same xAI probability function for local consistency.")
print("[INFO] Direct model probabilities are used when the notebook exposes them; otherwise a compact differentiable probability head is fitted to the notebook outputs.")
print("[INFO] Explained classes:", [class_names_xai[i] for i in explain_class_ids])

xai_head, X_std, xai_center, xai_scale, xai_device, proxy_fidelity = _xai_train_probability_head(X_view, proba)
print("[INFO] xAI probability-head fidelity:", proxy_fidelity)


def _xai_probability_fn(x_batch):
    return _xai_predict_from_head(xai_head, xai_center, xai_scale, xai_device, x_batch)

if SHAP_FINANCIAL_ENABLED:
    attrs_by_class, gradientshap_method = _xai_gradientshap_attributions(
        xai_head, X_std, grad_sample_indices, background_indices, explain_class_ids, xai_device
    )
    global_rows = []
    for class_id, attrs in attrs_by_class.items():
        mean_abs = np.mean(np.abs(attrs), axis=0)
        signed = np.mean(attrs, axis=0)
        for rank, j in enumerate(np.argsort(-mean_abs), start=1):
            global_rows.append({
                "rank": int(rank), "model_key": XAI_MODEL_KEY, "model_label": XAI_MODEL_LABEL,
                "class_id": int(class_id), "class_name": class_names_xai[class_id],
                "feature": str(feature_cols[j]), "mean_abs_gradientshap": float(mean_abs[j]),
                "mean_signed_gradientshap": float(signed[j]), "feature_source": feature_source,
                "probability_source": proba_source, "gradientshap_method": gradientshap_method,
                "xai_method": "captum_gradient_shap" if "captum" in gradientshap_method else "expected_gradients_fallback",
                "xai_probability_source": proba_source,
                "proxy_mse": proxy_fidelity["proxy_mse"], "proxy_argmax_agreement": proxy_fidelity["proxy_argmax_agreement"],
            })
    gradient_global_df = pd.DataFrame(global_rows)
    shap_importance_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_shap_importance_by_class.csv"
    gradient_global_df.to_csv(shap_importance_path, index=False, encoding="utf-8-sig")
    shap_global = (
        gradient_global_df.groupby("feature", as_index=False)
        .agg(mean_abs_shap_value=("mean_abs_gradientshap", "mean"), mean_signed_shap_value=("mean_signed_gradientshap", "mean"))
        .sort_values("mean_abs_shap_value", ascending=False)
        .reset_index(drop=True)
    )
    shap_global["rank"] = np.arange(1, len(shap_global) + 1)
    shap_global["model_key"] = XAI_MODEL_KEY
    shap_global["model_label"] = XAI_MODEL_LABEL
    shap_global["feature_source"] = feature_source
    shap_global["probability_source"] = proba_source
    shap_global["xai_probability_source"] = proba_source
    shap_global_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_shap_global_importance.csv"
    shap_global.to_csv(shap_global_path, index=False, encoding="utf-8-sig")

    top_global = shap_global.head(20)
    plt.figure(figsize=(10, 7))
    plt.barh(top_global["feature"][::-1], top_global["mean_abs_shap_value"][::-1], color="#006D77")
    plt.title(f"{XAI_MODEL_LABEL} Captum GradientSHAP Importance", fontweight="bold")
    plt.xlabel(f"mean(|GradientSHAP attribution|) from {XAI_MODEL_LABEL} xAI probability calls")
    plt.tight_layout()
    shap_bar_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_shap_importance.png"
    plt.savefig(shap_bar_path, dpi=180, bbox_inches="tight")
    plt.show()

    beeswarm_features = shap_global.head(min(XAI_BEESWARM_TOP_FEATURES, len(shap_global)))["feature"].tolist()
    beeswarm_idx = [feature_cols.index(f) for f in beeswarm_features if f in feature_cols]
    if beeswarm_idx:
        beeswarm_rows = []
        for class_id, attrs in attrs_by_class.items():
            for local_pos, row_idx in enumerate(grad_sample_indices):
                for feat_idx in beeswarm_idx:
                    beeswarm_rows.append({
                        "class_name": class_names_xai[class_id],
                        "feature": feature_cols[feat_idx],
                        "feature_value": float(X_view[row_idx, feat_idx]),
                        "shap_value": float(attrs[local_pos, feat_idx]),
                    })
        beeswarm_df = pd.DataFrame(beeswarm_rows)
        plt.figure(figsize=(11, max(6, 0.42 * len(beeswarm_features))))
        y_positions = {feature: pos for pos, feature in enumerate(reversed(beeswarm_features))}
        for feature in beeswarm_features:
            subset = beeswarm_df[beeswarm_df["feature"] == feature]
            if subset.empty:
                continue
            jitter = np.random.default_rng(XAI_RANDOM_STATE).normal(0, 0.05, size=len(subset))
            plt.scatter(subset["shap_value"], y_positions[feature] + jitter, c=subset["feature_value"], cmap="coolwarm", s=38, alpha=0.82, edgecolors="white", linewidths=0.35)
        plt.yticks(list(y_positions.values()), list(y_positions.keys()))
        plt.axvline(0.0, color="#334155", linewidth=1.0, alpha=0.7)
        plt.title(f"{XAI_MODEL_LABEL} GradientSHAP Beeswarm", fontweight="bold")
        plt.xlabel("GradientSHAP attribution for predicted class")
        cbar = plt.colorbar()
        cbar.set_label(f"Feature value ({XAI_FINANCIAL_REDUCER} sequence view)")
        plt.tight_layout()
        beeswarm_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_gradientshap_beeswarm.png"
        plt.savefig(beeswarm_path, dpi=180, bbox_inches="tight")
        plt.savefig(_xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_shap_beeswarm.png", dpi=180, bbox_inches="tight")
        plt.show()

    local_rows = []
    print()
    print("=== Captum GradientSHAP: Why this class? ===")
    selected_attr_lookup = {}
    selected_attrs, selected_method = _xai_gradientshap_attributions(
        xai_head, X_std, selected_indices, background_indices, explain_class_ids, xai_device
    )
    for pos, idx in enumerate(selected_indices):
        pred_id = int(np.argmax(proba[idx]))
        pred_name = class_names_xai[pred_id]
        row_meta = meta.iloc[int(idx)].to_dict() if len(meta) > int(idx) else {"test_index": int(idx)}
        print()
        print(f"--- Test index {int(idx)} | {XAI_MODEL_LABEL} predicted class={pred_name} | predicted_proba={proba[idx, pred_id]:.4f} ---")
        for class_id in explain_class_ids:
            attr = selected_attrs[int(class_id)][pos]
            order = np.argsort(-np.abs(attr))[:min(XAI_LOCAL_TOP_FEATURES, len(feature_cols))]
            print(f"Why this class? class={class_names_xai[class_id]} | model_proba={proba[idx, class_id]:.4f}")
            for rank, j in enumerate(order, start=1):
                weight = float(attr[j])
                direction = "supports_explained_class" if weight > 0 else "opposes_explained_class"
                if rank <= 5:
                    print(f"  {direction:<26} | {feature_cols[j]:<45} | attribution={weight:+.5f}")
                local_rows.append({
                    **row_meta, "test_index": int(idx), "rank": int(rank),
                    "model_key": XAI_MODEL_KEY, "model_label": XAI_MODEL_LABEL,
                    "predicted_class_id": int(pred_id), "predicted_class": pred_name,
                    "predicted_probability": float(proba[idx, pred_id]),
                    "explained_class_id": int(class_id), "explained_class": class_names_xai[class_id],
                    "explained_class_model_probability": float(proba[idx, class_id]),
                    "feature": str(feature_cols[j]), "feature_value": float(X_view[idx, j]),
                    "gradientshap_attribution": weight, "direction": direction,
                    "feature_source": feature_source, "probability_source": proba_source,
                    "xai_probability_source": proba_source,
                    "xai_method": "captum_gradient_shap" if "captum" in selected_method else "expected_gradients_fallback",
                    "gradientshap_method": selected_method,
                    "proxy_mse": proxy_fidelity["proxy_mse"],
                    "proxy_argmax_agreement": proxy_fidelity["proxy_argmax_agreement"],
                })
    gradient_local_df = pd.DataFrame(local_rows)
    shap_local_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_shap_local_decisions.csv"
    gradient_local_df.to_csv(shap_local_path, index=False, encoding="utf-8-sig")
    print("\nTop Captum GradientSHAP features:")
    print(shap_global.head(12).to_string(index=False))
    print("Saved:", shap_importance_path)
    print("Saved:", shap_global_path)
    print("Saved:", shap_bar_path)
    if "beeswarm_path" in locals():
        print("Saved:", beeswarm_path)
    print("Saved:", shap_local_path)
else:
    print("Financial SHAP is disabled. Set SHAP_FINANCIAL_ENABLED=True to run explanations.")

if LIME_FINANCIAL_ENABLED:
    try:
        import lime
        import lime.lime_tabular
        lime_available = True
    except Exception:
        lime_available = False

    if lime_available:
        explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data=X_view[background_indices],
            feature_names=[str(c) for c in feature_cols],
            class_names=class_names_xai,
            mode="classification",
            discretize_continuous=True,
            random_state=int(XAI_RANDOM_STATE),
        )
        print("[INFO] LIME package available; using LimeTabularExplainer.")
    else:
        explainer = None
        print("[WARN] Package 'lime' is not available; using weighted local-linear LIME fallback.")

    lime_rows = []
    print()
    print("=== Direct LIME using notebook probability outputs ===")
    for idx in selected_indices:
        idx = int(idx)
        pred_id = int(np.argmax(proba[idx]))
        pred_name = class_names_xai[pred_id]
        row_meta = meta.iloc[idx].to_dict() if len(meta) > idx else {"test_index": idx}
        lime_proba = _xai_probability_fn(X_view[idx].reshape(1, -1))[0]
        print()
        print(f"--- Test index {idx} | {XAI_MODEL_LABEL} predicted class={pred_name} | model_proba={proba[idx, pred_id]:.4f} | lime_proba={lime_proba[pred_id]:.4f} ---")
        html_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_lime_test_idx_{idx}.html"
        exp = None
        if lime_available:
            exp = explainer.explain_instance(
                data_row=X_view[idx],
                predict_fn=_xai_probability_fn,
                num_features=min(XAI_LOCAL_TOP_FEATURES, len(feature_cols)),
                labels=explain_class_ids,
                num_samples=int(LIME_NUM_SAMPLES),
            )
            try:
                exp.save_to_file(str(html_path))
            except Exception as exc:
                print(f"[WARN] Could not save LIME HTML for idx={idx}: {exc}")
        lime_instance_plot_path = _xai_save_lime_multiclass_instance_plot(
            row_meta=row_meta, exp=exp, model_proba=proba[idx], lime_proba=lime_proba,
            x_row=X_view[idx], html_path=html_path, lime_available=lime_available,
        )
        for class_id in explain_class_ids:
            if lime_available and exp is not None:
                items = [(str(rule), float(weight)) for rule, weight in exp.as_list(label=int(class_id))]
                lime_source = "LimeTabularExplainer"
            else:
                items = _xai_lime_fallback(X_view[idx], _xai_probability_fn, int(class_id), feature_cols, min(XAI_LOCAL_TOP_FEATURES, len(feature_cols)))
                lime_source = "weighted_local_linear_fallback"
            print(f"Why this class? class={class_names_xai[class_id]} | model_proba={proba[idx, class_id]:.4f} | lime_proba={lime_proba[class_id]:.4f}")
            for rank, (rule, weight) in enumerate(items, start=1):
                direction = "supports_explained_class" if weight > 0 else "opposes_explained_class"
                if rank <= 5:
                    print(f"  {direction:<26} | {rule:<45} | weight={weight:+.5f}")
                lime_rows.append({
                    **row_meta, "test_index": idx, "rank": int(rank),
                    "model_key": XAI_MODEL_KEY, "model_label": XAI_MODEL_LABEL,
                    "predicted_class_id": int(pred_id), "predicted_class": pred_name,
                    "predicted_probability": float(proba[idx, pred_id]),
                    "explained_class_id": int(class_id), "explained_class": class_names_xai[class_id],
                    "explained_class_model_probability": float(proba[idx, class_id]),
                    "explained_class_lime_prediction_probability": float(lime_proba[class_id]),
                    "lime_rule": str(rule), "lime_weight_for_explained_class": float(weight),
                    "direction": direction, "feature_source": feature_source,
                    "probability_source": proba_source, "lime_probability_source": lime_source,
                    "xai_probability_source": proba_source,
                    "proxy_mse": proxy_fidelity["proxy_mse"],
                    "proxy_argmax_agreement": proxy_fidelity["proxy_argmax_agreement"],
                    "html_path": str(html_path) if lime_available else "",
                    "lime_instance_plot_path": str(lime_instance_plot_path) if lime_instance_plot_path is not None else "",
                })
    lime_local_df = pd.DataFrame(lime_rows)
    lime_local_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_lime_local_decisions.csv"
    lime_local_df.to_csv(lime_local_path, index=False, encoding="utf-8-sig")
    if not lime_local_df.empty:
        lime_plot_df = lime_local_df.copy()
        lime_plot_df["abs_weight"] = lime_plot_df["lime_weight_for_explained_class"].abs()
        lime_summary = (
            lime_plot_df.groupby(["lime_rule", "direction"], as_index=False)
            .agg(mean_abs_weight=("abs_weight", "mean"), mean_signed_weight=("lime_weight_for_explained_class", "mean"), count=("lime_rule", "size"))
            .sort_values("mean_abs_weight", ascending=False)
            .head(int(XAI_LIME_PLOT_TOP_RULES))
            .iloc[::-1]
        )
        colors = ["#0F766E" if w > 0 else "#B91C1C" for w in lime_summary["mean_signed_weight"]]
        labels = [str(rule) if len(str(rule)) <= 58 else str(rule)[:55] + "..." for rule in lime_summary["lime_rule"]]
        plt.figure(figsize=(11, max(6, 0.45 * len(lime_summary))))
        plt.barh(labels, lime_summary["mean_signed_weight"], color=colors, alpha=0.88)
        plt.axvline(0.0, color="#334155", linewidth=1.0)
        plt.title(f"{XAI_MODEL_LABEL} LIME Local Rule Summary", fontweight="bold")
        plt.xlabel("Mean LIME weight across selected test cases and explained classes")
        plt.tight_layout()
        lime_plot_path = _xai_artifact_dir() / f"{XAI_MODEL_KEY}_financial_lime_rule_summary.png"
        plt.savefig(lime_plot_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Saved:", lime_plot_path)
    print("Saved per-class local LIME decision explanations to:", lime_local_path)
else:
    print("LIME disabled. Set LIME_FINANCIAL_ENABLED=True to run.")
